# QF600 · Asset Pricing
## Group Project · Group 6 · Equity Risk Premium in a Higher-Rate World · `Execution Script`

> Do equities do poorly when risk-free rates are high, and what should an investor hold now? This notebook pulls the data, replicates Maloney (2024) "Honey, the Fed Shrunk the Equity Premium", tests it out of sample, and backtests the allocation rules S0–S6. Every number quoted in the report and slides is written to `outputs/results.json`; every figure to `figures/`.

## 👉 <a id="top">Table of contents</a> 👈

0. [Setup](#s0)
1. [Data & hypotheses](#s1)
2. [Replication of the paper](#s2)
3. [Profits & the deficit](#s3)
4. [Out-of-sample tests](#s4)
5. [What should an investor hold?](#s5)
6. [Backtests & simulation](#s6)
7. [Results export](#s7)

## <a id="s0">0 · Setup</a>

[↑ Contents](#top)

One cell for every import. The FRED key is read from `credentials.yaml`, which is gitignored. See README step 8. Raw files are saved to `datasets/` so the analysis can be re-run offline.

In [1]:
import io
import re
import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yaml
import yfinance as yf
from fredapi import Fred
import statsmodels.api as sm
from scipy.optimize import minimize

from lets_plot import *
LetsPlot.setup_html()

warnings.filterwarnings("ignore", category = FutureWarning)

# house palette (context/08): teal, blue, orange, gold, then muted greys
PALETTE = ["#5CC6A7", "#5B8DEF", "#E2692F", "#F0B64A", "#9EA6A3", "#C8CECB", "#3FA27A", "#34504A"]
FIGS = Path("figures")
FIGS.mkdir(exist_ok = True)

def save_fig(plot, fig_id, legend = None):
    """Save the report version (figures/<id>.png) and a slide version with big text and no titles
    (figures/<id>_slide.png): 30 px text on a 1100 px chart shows at about 12-13 pt on a 6.5-inch slide figure."""
    ggsave(plot, f"{fig_id}.png", path = str(FIGS.resolve()), scale = 2)
    slide = plot + theme(text = element_text(size = 30), axis_title = element_text(size = 28), plot_title = element_blank(),
                         plot_subtitle = element_blank(), plot_caption = element_blank())
    if legend:
        slide = slide + theme(legend_position = legend)
    ggsave(slide, f"{fig_id}_slide.png", path = str(FIGS.resolve()), scale = 2)

creds = yaml.safe_load(open("credentials.yaml"))
DATA = Path("datasets")
DATA.mkdir(exist_ok = True)

# results registry: every number quoted in the report/slides goes through record()
results = {}
def record(key, value, fmt = "{:.1%}", note = ""):
    results[key] = {"value": float(value), "display": fmt.format(value), "note": note}
    return value

## <a id="s1">1 · Data & hypotheses</a>

[↑ Contents](#top)

#### Walkthrough step 1: collect the data and write down our predictions

The paper's long dataset is **monthly, Jan 1926 – Dec 2023** (Maloney 2024, p.4–5, "Long Dataset"). We extend it to the latest month. Where the paper used paid sources (Global Financial Data, Bloomberg), we use free proxies, listed in `context/06_data_sources.md`.

| § | Source | What | Frequency | From | API key |
|---|--------|------|-----------|------|---------|
| 1.1 | FRED | 3M T-bill (TB3MS, DGS3MO), 10Y yield, Aaa/Baa yields, deficit, interest outlays, CPI, GDP | monthly / daily / annual | 1913–1981 | `fred_key` |
| 1.2 | Ken French | Market excess return, RF (1M T-bill), SMB, HML, momentum (UMD) | monthly | Jul 1926 | — |
| 1.3 | Robert Shiller | S&P price, dividends, earnings, CPI, real EPS, CAPE | monthly | 1871 | — |
| 1.4 | Damodaran | Annual returns: S&P 500, 3M T-bill, 10Y T-bond, Baa corporate, real estate, gold | annual | 1928 | — |
| 1.5 | Yahoo Finance | ETF prices SPY, IEF, TLT, HYG, VNQ, PSP, GLD, RSP, PDP | daily | 2002 | — |
| 1.6 | World Bank Pink Sheet | Monthly gold price ($/troy oz) | monthly | 1960 | — |

### 1.1 FRED

FRED quotes rates in **annualised %**. We keep the raw % column and add a decimal column. `fred_series()` downloads one series and returns a tidy frame with a `Date` index.

In [2]:
fred = Fred(api_key = creds["fred_key"])

def fred_series(series_id, name = None, cache = None):
    """Download one FRED series. If FRED is down, fall back to the last saved file in datasets/."""
    name = name or series_id
    try:
        return (
            fred
            .get_series(series_id)
            .dropna()
            .rename_axis("Date")
            .to_frame(name)
        )
    except Exception as e:
        if cache and (DATA / cache).exists():
            print(f"FRED unavailable for {series_id} ({type(e).__name__}): using saved {cache}")
            return pd.read_csv(DATA / cache, index_col = "Date", parse_dates = ["Date"])[[name]]
        raise

In [3]:
# Monthly 3M T-bill rate, secondary market (FRED quotes in annualised %); from Jan 1934
TB3MS =\
(
    fred_series("TB3MS", "TB3MS_pct", cache = "fred_tb3ms.csv")
    .assign(TB3MS = lambda d: d["TB3MS_pct"] / 100)
)
TB3MS.to_csv(DATA / "fred_tb3ms.csv")
TB3MS

,TB3MS_pct,TB3MS
Date,,
1934-01-01,0.72,0.0072
1934-02-01,0.62,0.0062
1934-03-01,0.24,0.0024
1934-04-01,0.15,0.0015
1934-05-01,0.16,0.0016
...,...,...
2026-05-01,3.60,0.0360
2026-06-01,3.66,0.0366
2026-07-01,3.73,0.0373


In [4]:
# Daily 3M Treasury constant-maturity yield (annualised %); from Sep 1981. Used for recent daily checks
DGS3MO =\
(
    fred_series("DGS3MO", "DGS3MO_pct", cache = "fred_dgs3mo.csv")
    .assign(DGS3MO = lambda d: d["DGS3MO_pct"] / 100)
)
DGS3MO.to_csv(DATA / "fred_dgs3mo.csv")
DGS3MO

,DGS3MO_pct,DGS3MO
Date,,
1981-09-01,17.01,0.1701
1981-09-02,16.65,0.1665
1981-09-03,16.96,0.1696
1981-09-04,16.64,0.1664
1981-09-08,16.54,0.1654
...,...,...
2026-10-23,4.22,0.0422
2026-10-26,4.22,0.0422
2026-10-27,4.22,0.0422


In [5]:
# 10Y Treasury yield (from 1953) and Moody's Aaa / Baa corporate yields (from 1919), monthly, %
GS10 = fred_series("GS10", cache = "fred_gs10.csv")
GS10.to_csv(DATA / "fred_gs10.csv")

AAA_BAA =\
(
    fred_series("AAA", cache = "fred_aaa_baa.csv")
    .join(fred_series("BAA", cache = "fred_aaa_baa.csv"), how = "outer")
)
AAA_BAA.to_csv(DATA / "fred_aaa_baa.csv")
AAA_BAA.tail()

,AAA,BAA
Date,,
2026-05-01,5.56,6.10
2026-06-01,5.52,6.00
2026-07-01,5.76,6.19
2026-08-01,5.88,6.32
2026-09-01,6.03,6.47


In [6]:
# Federal deficit for H3 (walkthrough step 3). Annual, fiscal year; surplus is positive, deficit negative
deficit_gdp  = fred_series("FYFSGDA188S", "surplus_pct_gdp", cache = "fred_deficit_gdp.csv")    # 1929 →
interest_gdp = fred_series("FYOIGDA188S", "interest_pct_gdp", cache = "fred_interest_gdp.csv")   # 1940 →, for the primary deficit
mts_deficit  = fred_series("MTSDS133FMS", "surplus_musd", cache = "fred_mts_deficit.csv")       # monthly, $ millions, Oct 1980 →
gdp          = fred_series("GDPA", "gdp_busd", cache = "fred_gdp.csv")                  # annual nominal GDP, $ billions

deficit_gdp.to_csv(DATA / "fred_deficit_gdp.csv")
interest_gdp.to_csv(DATA / "fred_interest_gdp.csv")
mts_deficit.to_csv(DATA / "fred_mts_deficit.csv")
gdp.to_csv(DATA / "fred_gdp.csv")

# CPI (all urban, not seasonally adjusted), monthly from 1913: for real returns
CPI = fred_series("CPIAUCNS", "CPI", cache = "fred_cpi.csv")
CPI.to_csv(DATA / "fred_cpi.csv")

deficit_gdp.join(interest_gdp).tail()

,surplus_pct_gdp,interest_pct_gdp
Date,,
2021-01-01,-11.67437,2.36836
2022-01-01,-5.27784,2.75372
2023-01-01,-6.09198,3.16003
2024-01-01,-6.23810,3.85636
2025-01-01,-5.75270,3.93896


### 1.2 Ken French data library

Monthly factors in **% per month** (Lecture 4, slide 19). `RF` is the 1-month T-bill return; before FRED's TB3MS starts (1934) we use it to extend the T-bill series back to 1926, which the paper did with Global Financial Data. Each zip holds one CSV with a text header, a monthly block and an annual block; `kf_monthly()` keeps the monthly block.

In [7]:
def kf_monthly(name):
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{name}_CSV.zip"
    z = zipfile.ZipFile(io.BytesIO(requests.get(url, timeout = 60).content))
    lines = z.read(z.namelist()[0]).decode("latin-1").splitlines()
    start = next(i for i, l in enumerate(lines) if l.startswith(","))      # header row: ",Mkt-RF,..."
    end = next(i for i in range(start + 1, len(lines)) if not lines[i].strip())
    return (
        pd
        .read_csv(io.StringIO("\n".join(lines[start:end])), index_col = 0)
        .rename_axis("Date")
        .pipe(lambda d: d.set_axis(pd.to_datetime(d.index.astype(str).str.strip(), format = "%Y%m"), axis = 0))
        .apply(pd.to_numeric, errors = "coerce")
        .replace([-99.99, -999], np.nan)
    )

FF3 = kf_monthly("F-F_Research_Data_Factors")        # Mkt-RF, SMB, HML, RF   (Jul 1926 →)
UMD = kf_monthly("F-F_Momentum_Factor")              # Mom                     (Jan 1927 →)
UMD.columns = ["UMD"]

FF3.to_csv(DATA / "ff3_monthly.csv")
UMD.to_csv(DATA / "ff_umd_monthly.csv")
FF3.join(UMD).tail()

,Mkt-RF,SMB,HML,RF,UMD
Date,,,,,
2026-05-01,4.91,-1.59,-2.31,0.31,1.39
2026-06-01,-1.07,3.39,3.58,0.29,4.62
2026-07-01,-0.61,-1.92,2.11,0.33,-12.21
2026-08-01,2.56,0.34,-3.54,0.29,-5.60
2026-09-01,-1.07,-3.16,-1.55,0.31,6.70


### 1.3 Robert Shiller: earnings and CAPE

The paper uses Shiller's real EPS (p.6, "blistering 11%") and CAPE (p.7, "two-factor regression"). The Yale copy of `ie_data.xls` stopped updating in 2023, so we take the current file from shillerdata.com. Its link changes with every update, so we find it on the page.

Shiller writes dates as `YYYY.MM` numbers, so **October is `2026.1`**. We convert them with year = integer part and month = round(fraction × 100).

In [8]:
page = requests.get("https://shillerdata.com/", timeout = 30, headers = {"User-Agent": "Mozilla/5.0"}).text
href = re.search(r'href="([^"]*ie_data\.xls[^"]*)"', page).group(1)
shiller_url = "https:" + href if href.startswith("//") else href
(DATA / "shiller_ie_data.xls").write_bytes(requests.get(shiller_url, timeout = 60).content)

shiller_cols = {0: "date_raw", 1: "P", 2: "D", 3: "E", 4: "CPI", 6: "GS10",
                7: "real_P", 8: "real_D", 10: "real_E", 12: "CAPE"}

SHILLER =\
(
    pd
    .read_excel(DATA / "shiller_ie_data.xls", sheet_name = "Data", header = None, skiprows = 8)
    .loc[:, list(shiller_cols)]
    .rename(columns = shiller_cols)
    .assign(date_raw = lambda d: pd.to_numeric(d["date_raw"], errors = "coerce"))
    .dropna(subset = ["date_raw"])
    .assign(Date = lambda d: pd.to_datetime(dict(year = d["date_raw"].astype(int),
                                                 month = ((d["date_raw"] % 1) * 100).round().astype(int),
                                                 day = 1)))
    .set_index("Date")
    .drop(columns = "date_raw")
    .apply(pd.to_numeric, errors = "coerce")
)
SHILLER.to_csv(DATA / "shiller_monthly.csv")
SHILLER.tail()

,P,D,E,CPI,GS10,real_P,real_D,real_E,CAPE
Date,,,,,,,,,
2026-06-01,7450.032857,81.7032,295.3881,333.9520,4.47,7490.735069,82.149574,297.001912,40.150170
2026-07-01,7481.339091,NaN,NaN,333.9180,4.60,7522.978262,NaN,NaN,40.008636
2026-08-01,7711.323333,NaN,NaN,334.9800,4.68,7729.658962,NaN,NaN,40.987394
2026-09-01,7669.414286,NaN,NaN,335.5110,4.99,7675.483325,NaN,NaN,40.580874
2026-10-01,7722.720000,NaN,NaN,335.7765,5.24,7722.720000,NaN,NaN,40.710609


### 1.4 Damodaran: annual returns 1928 → latest

Our first-pass check (walkthrough step 2: 9.1% vs 6.7% above cash) and the bond/credit cross-check (step 1) use Damodaran's yearly returns. Sheet `Returns by year`: the header row starts with `Year`; the first 8 columns are the annual returns.

In [9]:
damodaran_url = "https://pages.stern.nyu.edu/~adamodar/pc/datasets/histretSP.xls"
(DATA / "damodaran_histretSP.xls").write_bytes(requests.get(damodaran_url, timeout = 60).content)

raw = pd.read_excel(DATA / "damodaran_histretSP.xls", sheet_name = "Returns by year", header = None)
header_row = raw.index[raw[0].astype(str).str.strip().eq("Year")][0]

DAMODARAN =\
(
    raw
    .iloc[header_row + 1:, :8]
    .set_axis(["Year", "SP500", "SmallCap", "TBill_3M", "TBond_10Y", "Baa_Corp", "RealEstate", "Gold"], axis = 1)
    .assign(Year = lambda d: pd.to_numeric(d["Year"], errors = "coerce"))
    .dropna(subset = ["Year"])
    .astype({"Year": int})
    .set_index("Year")
    .apply(pd.to_numeric, errors = "coerce")
)
DAMODARAN.to_csv(DATA / "damodaran_annual.csv")
DAMODARAN.tail()

,SP500,SmallCap,TBill_3M,TBond_10Y,Baa_Corp,RealEstate,Gold
Year,,,,,,,
2021,0.284689,0.2241,0.0004,-0.044160,0.010161,0.188646,-0.037544
2022,-0.180375,-0.2290,0.0209,-0.178282,-0.152281,0.056518,0.005494
2023,0.260607,0.0519,0.0528,0.038800,0.087357,0.056784,0.132621
2024,0.248786,0.0870,0.0518,-0.016372,0.017364,0.039634,0.259570
2025,0.177237,0.1653,0.0421,0.077955,0.069627,0.015757,0.662158


### 1.5 ETF prices (Yahoo Finance)

For the Exhibit 9 portfolios and strategies S0–S6 (walkthrough step 6): stocks SPY, Treasuries IEF/TLT, high yield HYG, real estate VNQ, listed private equity PSP, gold GLD, equal-weight S&P RSP, and momentum PDP (Invesco Dorsey Wright Momentum, from Mar 2007). Prices are dividend-adjusted (`auto_adjust = True`). PDP starts Mar 2007 and HYG Apr 2007, so the step 6 ETF backtest starts in May 2007 (first full month with every fund).

In [10]:
etf_tickers = ["SPY", "IEF", "TLT", "HYG", "VNQ", "PSP", "GLD", "RSP", "PDP"]

ETF =\
(
    yf
    .download(etf_tickers, start = "2002-01-01", auto_adjust = True, progress = False)
    ["Close"]
    .rename_axis("Date")
)
ETF.to_csv(DATA / "etf_prices_daily.csv")
ETF.apply(lambda s: s.first_valid_index().date()).rename("first date").to_frame()

,first date
Ticker,
GLD,2004-11-18
HYG,2007-04-11
IEF,2002-07-30
PDP,2007-03-01
PSP,2006-10-24
RSP,2003-05-01
SPY,2002-01-02
TLT,2002-07-30
VNQ,2004-09-29


### 1.6 World Bank gold price (monthly, from 1960)

Gold is our diversifier in step 5, which uses the **longest sample the data allows**. The World Bank "Pink Sheet" has a monthly gold price ($/troy oz) from Jan 1960, free and with no key. The download link changes with each update, so we find it on the commodity-markets page.

Note: until Aug 1971 the US fixed gold at $35/oz (Bretton Woods), so gold barely moves in 1960–71. We keep those years to use the full sample, and say so in the report.

In [11]:
wb_page = requests.get("https://www.worldbank.org/en/research/commodity-markets", timeout = 30).text
wb_url = re.search(r'https://thedocs\.worldbank\.org[^"]*CMO-Historical-Data-Monthly\.xlsx', wb_page).group(0)
(DATA / "wb_cmo_monthly.xlsx").write_bytes(requests.get(wb_url, timeout = 120).content)

wb_raw = pd.read_excel(DATA / "wb_cmo_monthly.xlsx", sheet_name = "Monthly Prices", header = None)
first = wb_raw.index[wb_raw[0].astype(str).str.fullmatch(r"\d{4}M\d{2}")][0]     # first data row, e.g. "1960M01"
header = wb_raw.iloc[first - 3:first].astype(str).agg(" ".join)
gold_col = next(j for j, h in enumerate(header) if "Gold" in h)

GOLD =\
(
    wb_raw
    .iloc[first:, [0, gold_col]]
    .set_axis(["month", "gold_usd_oz"], axis = 1)
    .dropna()
    .assign(Date = lambda d: pd.to_datetime(d["month"], format = "%YM%m"),
            gold_usd_oz = lambda d: pd.to_numeric(d["gold_usd_oz"], errors = "coerce"))
    .set_index("Date")
    [["gold_usd_oz"]]
)
GOLD.to_csv(DATA / "wb_gold_monthly.csv")
GOLD.loc[pd.to_datetime(["1960-01-01", "1971-07-01", "1980-01-01"]).append(GOLD.index[-1:])]

,gold_usd_oz
1960-01-01,35
1971-07-01,41
1980-01-01,675
2026-09-01,4319


### 1.7 Global Financial Data (GFD): not used

SMU has no subscription to Finaeon / Global Financial Data, the paper's source, and the free trial only covers US stocks from 2019. We use free proxies over the paper's sample (monthly, Jan 1926 – Dec 2023):

| Paper's GFD series | Our proxy |
|--------------------|-----------|
| 3M T-bill rate, 1926–33 | Ken French RF (1-month T-bill) × 12, then FRED TB3MS from 1934 |
| US equity returns | Ken French market return (Mkt-RF + RF) |
| US Treasury returns | Damodaran annual 10Y T-bond return; monthly approximation from FRED GS10 |
| IG credit (paper: Asvanunt & Richardson) | Damodaran annual Baa corporate return; FRED Aaa/Baa yields |

### 1.8 Monthly master panel

One monthly frame for steps 2–6. The **T-bill rate** is TB3MS from 1934, extended back to Jul 1926 with Ken French RF × 12 (1-month bill, annualised). This matches the paper's method of splicing in another short-rate source before 1934 (p.4). **Equity excess return** = Mkt-RF; **momentum (cash-plus proxy)** = RF + UMD (H2).

**10-year Treasury return.** The paper used GFD bond returns; we build a monthly return from the 10-year yield, as Damodaran does for his annual series. Each month we hold a 10-year bond bought at par last month (coupon = last month's yield) and reprice it at this month's yield with 10 years minus one month left. The yield is FRED GS10 from Apr 1953 and Shiller's long-rate column before that.

In [12]:
def bond_return(yield_pct, maturity = 10):
    """Monthly total return of a par bond bought last month, repriced at this month's yield (semi-annual coupons)."""
    y0, y1 = yield_pct.shift(1) / 100, yield_pct / 100
    n = 2 * (maturity - 1 / 12)                                   # semi-annual periods left
    price = (y0 / y1) * (1 - (1 + y1 / 2) ** -n) + (1 + y1 / 2) ** -n
    return (price - 1 + y0 / 12).rename("bond10")

y10 = GS10["GS10"].combine_first(SHILLER["GS10"].loc[:"1953-03"])
bond10 = bond_return(y10)

gold_ret = GOLD["gold_usd_oz"].pct_change().rename("gold")

tbill =\
(
    TB3MS["TB3MS_pct"]
    .combine_first(FF3["RF"].mul(12).loc[:"1933-12"])     # pre-1934 back-fill from the 1-month bill
    .rename("tbill_pct")
)

panel =\
(
    pd
    .concat([tbill,
             FF3[["Mkt-RF", "RF", "SMB", "HML"]],
             UMD,
             GS10,
             AAA_BAA,
             CPI,
             SHILLER[["real_E", "CAPE"]],
             bond10,
             gold_ret],
            axis = 1)
    .loc["1926-07":]
    .assign(eq_xret  = lambda d: d["Mkt-RF"] / 100,
            rf_m     = lambda d: d["RF"] / 100,
            equity   = lambda d: (d["Mkt-RF"] + d["RF"]) / 100,
            mom_cash = lambda d: (d["RF"] + d["UMD"]) / 100)
)
panel.tail()

,tbill_pct,Mkt-RF,RF,SMB,HML,UMD,GS10,AAA,BAA,CPI,real_E,CAPE,bond10,gold,eq_xret,rf_m,equity,mom_cash
Date,,,,,,,,,,,,,,,,,,
2026-06-01,3.66,-1.07,0.29,3.39,3.58,4.62,4.47,5.52,6.00,333.952,297.001912,40.150170,0.004527,-0.078265,-0.0107,0.0029,-0.0078,0.0491
2026-07-01,3.73,-0.61,0.33,-1.92,2.11,-12.21,4.60,5.76,6.19,333.918,NaN,40.008636,-0.006534,-0.036660,-0.0061,0.0033,-0.0028,-0.1188
2026-08-01,3.72,2.56,0.29,0.34,-3.54,-5.60,4.68,5.88,6.32,334.980,NaN,40.987394,-0.002456,0.082986,0.0256,0.0029,0.0285,-0.0531
2026-09-01,3.94,-1.07,0.31,-3.16,-1.55,6.70,4.99,6.03,6.47,336.940,NaN,40.580874,-0.020118,-0.020857,-0.0107,0.0031,-0.0076,0.0701
2026-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.710609,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Check our bond return against Damodaran's published annual 10Y T-bond return
bond_check =\
(
    pd
    .DataFrame({"ours": (1 + bond10).groupby(bond10.index.year).prod() - 1,
                "damodaran": DAMODARAN["TBond_10Y"]})
    .loc[1928:2025]
    .dropna()
)
print(f"correlation {bond_check.corr().iloc[0, 1]:.3f} · mean ours {bond_check['ours'].mean():.2%} "
      f"vs Damodaran {bond_check['damodaran'].mean():.2%} · {len(bond_check)} years")
record("bond_check_corr", bond_check.corr().iloc[0, 1], "{:.2f}", "annual corr, our 10Y return vs Damodaran")
bond_check.tail()

correlation 0.986 · mean ours 5.05% vs Damodaran 4.82% · 98 years


,ours,damodaran
2021,-0.036232,-0.044160
2022,-0.149231,-0.178282
2023,0.005488,0.038800
2024,0.011376,-0.016372
2025,0.064796,0.077955


In [14]:
# Data inventory: what was saved, how many rows, and the date range of each file
inventory =\
(
    pd
    .DataFrame([{"file": p.name,
                 "rows": len(pd.read_csv(p, index_col = 0)),
                 "first": pd.read_csv(p, index_col = 0).index.min(),
                 "last":  pd.read_csv(p, index_col = 0).index.max()}
                for p in sorted(DATA.glob("*.csv"))])
    .set_index("file")
)
inventory

,rows,first,last
file,,,
damodaran_annual.csv,98,1928,2025
etf_prices_daily.csv,6248,2002-01-02,2026-10-30
ff3_monthly.csv,1203,1926-07-01,2026-09-01
ff_umd_monthly.csv,1197,1927-01-01,2026-09-01
fred_aaa_baa.csv,1293,1919-01-01,2026-09-01
fred_cpi.csv,1364,1913-01-01,2026-09-01
fred_deficit_gdp.csv,97,1929-01-01,2025-01-01
fred_dgs3mo.csv,11290,1981-09-01,2026-10-29
fred_gdp.csv,97,1929-01-01,2025-01-01


After running this section, ask Claude to **"sync the metadata"** (`qf600-data-sync`). It profiles these files and updates `02 - Dataset Metadata Group 6.md` and the report's Annex A.

### Hypotheses (registered before looking at results)

**Part A · Replicate the paper** on the paper's sample (monthly, Jan 1926 – Dec 2023), using the free proxies above for its Global Financial Data series.

| ID | Hypothesis | Source |
|----|-----------|--------|
| **H0** | The level of the T-bill rate makes no difference: stocks earn the same above cash after high-rate months as after low-rate months. | Null hypothesis |
| **H1** | After high-rate months, stocks earn less above cash (the paper's claim). | Maloney 2024, Key Findings p.1 |

**Part B · Test whether the paper's conclusion holds up**, using the Sharpe ratio and the fiscal deficit.

| ID | Hypothesis | Source |
|----|-----------|--------|
| **H2** | Sharpe ratio: once risk is counted, high rates don't hurt stocks much. The equity Sharpe ratio after high-rate months is not materially lower than after low-rate months (paper: 0.57 vs 0.31). Tested on 1926–2023 and on the extension to 2026. | Ours, challenging Maloney 2024 p.6 |
| **H3** | Fiscal-deficit offset: when the government runs a big deficit, company profits keep growing even with high rates, so stocks' return above cash after high-rate months is not lower when the deficit is big. | Ours (Kalecki profits identity) |

The paper's momentum claim (momentum earns about the same above cash in both regimes) is checked as part of the §2 replication.

## <a id="s2">2 · Replication of the paper (H0 vs H1) and the Sharpe ratio (H2)</a>

[↑ Contents](#top)

Part A of the hypotheses, on the paper's sample (monthly, Jul 1926 – Dec 2023), with our free proxies for its Global Financial Data series.

### 2.1 Higher- and lower-rate regimes (the paper's three rules)

(Maloney 2024, p.4 "Rolling categorization"; fn 4.) A month is **higher-rate** when the T-bill rate is above:
- **M1:** the median of Jul 1926 – Dec 2023 (the paper's sample);
- **M2:** the 40th percentile of the previous 5 years (60 months), the only rule an investor could have used at the time, so **strategies use M2 only**;
- **M3:** the 40th percentile of the 5 years centred on the month (needs future data, description only).

For the rolling rules, any rate below 0.5% is always lower. For strategies we use the label from the **end of the previous month** (`regime_m2_lag`), so there is no look-ahead.

In [15]:
def rolling_regime(rate, window = 60, q = 0.40, centred = False, floor = 0.5):
    if centred:
        cut = rate.rolling(window, center = True, min_periods = window).quantile(q)
    else:
        cut = rate.shift(1).rolling(window, min_periods = window).quantile(q)
    return ((rate > cut) & (rate >= floor)).astype(float).where(cut.notna() & rate.notna())   # 1 = higher, 0 = lower

rate = panel["tbill_pct"]
regimes =\
(
    pd
    .DataFrame({"m1": (rate > rate.loc[:"2023-12"].median()).astype(float).where(rate.notna()),
                "m2": rolling_regime(rate),
                "m3": rolling_regime(rate, centred = True)})
    .assign(regime_m2_lag = lambda d: d["m2"].shift(1))
)
print("Dec 2023 higher-rate under all three rules (paper p.5):", bool(regimes.loc["2023-12-01", ["m1", "m2", "m3"]].all()))
regimes.mean().rename("share of months higher-rate").to_frame()

Dec 2023 higher-rate under all three rules (paper p.5): True


,share of months higher-rate
m1,0.512884
m2,0.501312
m3,0.525350
regime_m2_lag,0.501312


In [16]:
fig01_df =\
(
    pd.DataFrame({"tbill": rate, "regime": regimes["m2"].map({1.0: "Higher-rate (M2)", 0.0: "Lower-rate (M2)"})})
    .dropna().rename_axis("Date").reset_index()
)
fig01 =\
(
    ggplot(fig01_df, aes(x = "Date", y = "tbill"))
    + geom_line(color = "#9EA6A3", size = 0.6)
    + geom_point(aes(color = "regime"), size = 1.2)
    + scale_color_manual(values = [PALETTE[2], PALETTE[0]])
    + labs(title = "A century of interest-rate regimes",
           subtitle = "3-month T-bill rate, coloured by the paper's trailing 5-year rule (M2), Jul 1931 – Sep 2026",
           caption = "Source: FRED TB3MS, Ken French RF; Group 6 calculations",
           x = "", y = "T-bill rate (%)", color = "")
    + theme(text = element_text(size = 14), legend_position = "top")
    + ggsize(1100, 450)
)
save_fig(fig01, "fig01_regimes")
record("today_tbill", rate.dropna().iloc[-1], "{:.2f}%"); record("today_regime_high", regimes["m2"].dropna().iloc[-1], "{:.0f}")
fig01

Fontconfig error: Cannot load default config file: No such file: (null)


### 2.2 Exhibit 3: return above cash after higher- and lower-rate months

For each month labelled higher or lower, we take the average return above cash over the **next 1, 12 and 36 months** (annualised), then average across months and across the three rules, as the paper does (p.5–6).
- **Equities:** Ken French Mkt-RF.
- **Treasuries:** 10-year bond return (§1.8) minus cash.
- **IG credit:** a Baa bond return built from Moody's Baa yield (same method) minus the 10-year Treasury return. This is a proxy for the paper's duration-matched credit excess return (Asvanunt & Richardson 2017).
- **Momentum:** UMD, which is already a return above cash because it is self-financing.

In [17]:
PAPER_END = "2023-12"
H_LIST = [1, 12, 36]

baa_ret = bond_return(AAA_BAA["BAA"])
xr =\
(
    pd
    .DataFrame({"Equities":   panel["eq_xret"],
                "Treasuries": panel["bond10"] - panel["rf_m"],
                "IG credit":  baa_ret.reindex(panel.index) - panel["bond10"],
                "Momentum":   panel["UMD"] / 100})
    .loc["1926-07":]
)

def fwd_mean(x, h):
    """Average of x over the next h months (t+1 … t+h), placed at month t."""
    return x.rolling(h).mean().shift(-h)

def exhibit3(x, labels, h_list = H_LIST):
    out = {}
    for h in h_list:
        f = fwd_mean(x, h) * 12
        for rule in labels:
            for g, lab in [(1, "Higher"), (0, "Lower")]:
                out[(h, rule, lab)] = f[labels[rule] == g].mean()
    return pd.Series(out)

paper_x = xr.loc[:PAPER_END]
rules = regimes[["m1", "m2", "m3"]].loc[:PAPER_END]
ex3 = {a: exhibit3(paper_x[a], rules) for a in xr.columns}
ex3_avg =\
(
    pd
    .DataFrame(ex3)
    .groupby(level = [0, 2]).mean()          # average across the three rules
    .rename_axis(["horizon", "regime"])
)

paper_ex3 = pd.DataFrame({            # Maloney (2024) Exhibit 3, Panel A (avg of 3 rules)
    "Equities":   [11.4, 4.9, 10.3, 5.8, 10.4, 5.5],
    "Treasuries": [2.2, 0.6, 1.5, 1.2, 1.7, 1.2],
    "IG credit":  [3.0, 0.7, 2.6, 1.1, 2.3, 1.3]},
    index = pd.MultiIndex.from_product([H_LIST, ["Lower", "Higher"]], names = ["horizon", "regime"])) / 100

for a in ["Equities", "Treasuries", "IG credit", "Momentum"]:
    for h in H_LIST:
        for g in ["Lower", "Higher"]:
            record(f"ex3_{a.split()[-1].lower()}_{g.lower()}_{h}m", ex3_avg.loc[(h, g), a], "{:.1%}")
compare3 = ex3_avg.join(paper_ex3, rsuffix = " (paper)")
compare3[sorted(compare3.columns)].round(3)

Equities  Equities (paper)  IG credit  IG credit (paper)  \
horizon regime                                                             
1       Higher     0.043             0.049      0.002              0.007   
        Lower      0.120             0.114      0.040              0.030   
12      Higher     0.054             0.058      0.002              0.011   
        Lower      0.111             0.103      0.040              0.026   
36      Higher     0.062             0.055      0.010              0.013   
        Lower      0.104             0.104      0.031              0.023   

                Momentum  Treasuries  Treasuries (paper)  
horizon regime                                            
1       Higher     0.109       0.011               0.006  
        Lower      0.030       0.022               0.022  
12      Higher     0.099       0.019               0.012  
        Lower      0.040       0.015               0.015  
36      Higher     0.091       0.018               0.012  
        Lower      0.051       0.017               0.017

In [18]:
fig02_df =\
(
    ex3_avg[["Equities", "Treasuries", "IG credit"]]
    .stack().rename("ours").to_frame()
    .join(paper_ex3.stack().rename("paper"))
    .reset_index()
    .rename(columns = {"level_2": "asset"})
    .assign(horizon = lambda d: d["horizon"].astype(str) + "M",
            regime = lambda d: pd.Categorical(d["regime"] + " rates", ["Lower rates", "Higher rates"]))
)
fig02 =\
(
    ggplot(fig02_df, aes(x = "horizon", y = "ours", fill = "regime"))
    + geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.75)
    + geom_point(aes(y = "paper", group = "regime"), position = position_dodge(width = 0.8),
                 shape = 23, size = 5, fill = "white", color = "black")
    + facet_wrap("asset", ncol = 3, scales = "free_y")
    + scale_fill_manual(values = [PALETTE[0], PALETTE[2]])
    + scale_y_continuous(format = ".1%")
    + labs(title = "Smaller premiums after higher-rate months: our replication of Exhibit 3",
           subtitle = "Return above cash over the next 1, 12, 36 months, Jul 1926 – Dec 2023 · bars = ours, diamonds = paper",
           caption = "Source: Ken French, FRED, Shiller; Maloney (2024) Exhibit 3; Group 6 calculations",
           x = "Horizon", y = "Annualised return above cash", fill = "")
    + theme(text = element_text(size = 14), legend_position = "top")
    + ggsize(1100, 450)
)
save_fig(fig02, "fig02_exhibit3_replication")
fig02

### 2.3 Testing H0 against H1

**H0:** stocks earn the same above cash after higher- and lower-rate months. **H1 (the paper):** they earn less after higher-rate months.
We regress the next-month (and next-12-month) equity return above cash on a higher-rate dummy. The slope is the gap (annualised). Rate levels persist, so we use **Newey-West** standard errors (the paper's own caveat, p.5). H1 is one-sided, so we reject H0 when t < −1.645.

In [19]:
def nw_ols(y, X, lags):
    d = pd.concat([y, X], axis = 1).dropna()
    return sm.OLS(d.iloc[:, 0], sm.add_constant(d.iloc[:, 1:])).fit(cov_type = "HAC", cov_kwds = {"maxlags": lags})

h_rows = []
eq_p = paper_x["Equities"]
for h, lags in [(1, 6), (12, 18)]:
    y = fwd_mean(eq_p, h) * 12
    for rule in ["m1", "m2", "m3"]:
        fit = nw_ols(y, rules[rule].rename("higher"), lags)
        h_rows.append({"horizon": f"{h}M", "rule": rule, "gap (higher − lower)": fit.params["higher"],
                       "NW t": fit.tvalues["higher"], "one-sided p": fit.pvalues["higher"] / 2 if fit.params["higher"] < 0 else 1 - fit.pvalues["higher"] / 2,
                       "reject H0 (5%)": fit.tvalues["higher"] < -1.645})
h0_table = pd.DataFrame(h_rows).set_index(["horizon", "rule"])
for (h, rule), row in h0_table.iterrows():
    record(f"h0_gap_{h.lower()}_{rule}", row["gap (higher − lower)"], "{:.1%}")
    record(f"h0_t_{h.lower()}_{rule}", row["NW t"], "{:.2f}")
record("h0_rejections", h0_table["reject H0 (5%)"].sum(), "{:.0f}", "number of the 6 tests (2 horizons × 3 rules) that reject H0")
h0_table.round(3)

gap (higher − lower)   NW t  one-sided p  reject H0 (5%)
horizon rule                                                          
1M      m1                  -0.039 -1.010        0.156           False
        m2                  -0.063 -1.647        0.050            True
        m3                  -0.127 -3.182        0.001            True
12M     m1                  -0.059 -2.013        0.022            True
        m2                  -0.038 -1.430        0.076           False
        m3                  -0.075 -2.735        0.003            True

### 2.4 The paper's regression: does today's T-bill rate predict next month's return above cash?

Paper (p.5): t = −2.2 for equities and −2.4 for credit; not significant for Treasuries. We show the plain OLS t-statistic (comparable with the paper) and the Newey-West one.

In [20]:
reg_rows = []
tb = rate.loc[:PAPER_END].rename("tbill")
for a in ["Equities", "Treasuries", "IG credit"]:
    y = paper_x[a].shift(-1)
    d = pd.concat([y, tb], axis = 1).dropna()
    ols = sm.OLS(d.iloc[:, 0], sm.add_constant(d["tbill"])).fit()
    nw = nw_ols(d.iloc[:, 0], d[["tbill"]], 12)
    reg_rows.append({"asset": a, "slope (per 1pp, annualised)": ols.params["tbill"] * 12,
                     "OLS t": ols.tvalues["tbill"], "NW t": nw.tvalues["tbill"], "months": len(d)})
reg_table = pd.DataFrame(reg_rows).set_index("asset")
reg_table["paper t"] = [-2.2, np.nan, -2.4]
for a, key in [("Equities", "eq"), ("Treasuries", "tsy"), ("IG credit", "credit")]:
    record(f"reg_t_ols_{key}", reg_table.loc[a, "OLS t"], "{:.2f}")
    record(f"reg_t_nw_{key}", reg_table.loc[a, "NW t"], "{:.2f}")
reg_table.round(3)

,"slope (per 1pp, annualised)",OLS t,NW t,months,paper t
asset,,,,,
Equities,-0.013,-2.059,-2.069,1169,-2.2
Treasuries,-0.001,-0.299,-0.173,1169,NaN
IG credit,-0.005,-3.215,-2.900,1169,-2.4


### 2.5 H2: is the gap still there once risk is counted?

**H2 (ours):** the equity Sharpe ratio after higher-rate months is not materially lower than after lower-rate months. Paper: 0.57 vs 0.31 (p.6, "Economic drivers").
We compute monthly Sharpe ratios (annualised) by regime for each rule, on the paper's sample and on the extension to Sep 2026. A **block bootstrap** (12-month blocks, 2,000 draws) gives a 95% interval for the gap. Blocks keep the persistence of rates intact.

In [21]:
rng = np.random.default_rng(600)

def sharpe_by_regime(x, labels):
    return {g: x[labels == v].mean() / x[labels == v].std() * np.sqrt(12) for g, v in [("lower", 0), ("higher", 1)]}

def h2_stats(x, labs):
    s = {rule: sharpe_by_regime(x, labs[rule]) for rule in ["m1", "m2", "m3"]}
    lo = np.mean([v["lower"] for v in s.values()]); hi = np.mean([v["higher"] for v in s.values()])
    return lo, hi

def block_bootstrap_gap(x, labs, n = 2000, block = 12):
    d = pd.concat([x.rename("x"), labs], axis = 1).dropna()
    T = len(d); starts = np.arange(T - block)
    gaps = []
    for _ in range(n):
        ix = np.concatenate([np.arange(s, s + block) for s in rng.choice(starts, T // block + 1)])[:T]
        b = d.iloc[ix].reset_index(drop = True)
        lo, hi = h2_stats(b["x"], b[["m1", "m2", "m3"]])
        gaps.append(lo - hi)
    return np.percentile(gaps, [2.5, 97.5]), np.mean(np.array(gaps) <= 0)

h2_rows = []
for label, end in [("Jul 1926 – Dec 2023 (paper)", PAPER_END), ("Jul 1926 – Sep 2026", None)]:
    x = xr["Equities"].loc[:end]
    rlabs = regimes[["m1", "m2", "m3"]].loc[x.index]
    lo, hi = h2_stats(x, rlabs)
    ci, p = block_bootstrap_gap(x, rlabs)
    h2_rows.append({"sample": label, "Sharpe after lower": lo, "Sharpe after higher": hi, "gap": lo - hi,
                    "95% CI low": ci[0], "95% CI high": ci[1], "share of draws with gap ≤ 0": p})
h2_table = pd.DataFrame(h2_rows).set_index("sample")
for k, row in zip(["paper", "ext"], h2_table.itertuples()):
    record(f"h2_sharpe_lower_{k}", row[1], "{:.2f}"); record(f"h2_sharpe_higher_{k}", row[2], "{:.2f}")
    record(f"h2_gap_{k}", row[3], "{:.2f}"); record(f"h2_ci_low_{k}", row[4], "{:.2f}"); record(f"h2_ci_high_{k}", row[5], "{:.2f}")
h2_table.round(3)

,Sharpe after lower,Sharpe after higher,gap,95% CI low,95% CI high,share of draws with gap ≤ 0
sample,,,,,,
Jul 1926 – Dec 2023 (paper),0.594,0.275,0.319,0.004,0.746,0.023
Jul 1926 – Sep 2026,0.595,0.298,0.297,-0.017,0.723,0.032


In [22]:
fig03_df =\
(
    h2_table[["Sharpe after lower", "Sharpe after higher"]]
    .reset_index().melt(id_vars = "sample", var_name = "regime", value_name = "sharpe")
    .assign(regime = lambda d: pd.Categorical(d["regime"].str.replace("Sharpe after ", "After ") + " rates",
                                              ["After lower rates", "After higher rates"]))
)
fig03 =\
(
    ggplot(fig03_df, aes(x = "sample", y = "sharpe", fill = "regime"))
    + geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.75)
    + geom_text(aes(label = "sharpe"), label_format = ".2f", position = position_dodge(width = 0.8),
                vjust = -0.6, size = 7)
    + geom_hline(yintercept = 0, color = "#9EA6A3")
    + scale_fill_manual(values = [PALETTE[0], PALETTE[2]])
    + labs(title = "H2: stocks' Sharpe ratio after higher-rate months is about half",
           subtitle = "Monthly equity Sharpe ratio (annualised), average of the paper's three regime rules",
           caption = "Source: Ken French, FRED; Group 6 calculations. Paper: 0.57 vs 0.31",
           x = "", y = "Sharpe ratio", fill = "")
    + theme(text = element_text(size = 14), legend_position = "top")
    + ggsize(900, 450)
)
save_fig(fig03, "fig03_h2_sharpe")
fig03

### 2.6 Momentum check (the paper's cash-plus claim)

The paper says strategies that hold T-bills while trading earn about the same above cash in both regimes (Key Findings, p.1). For UMD the gap should be small and not significant.

In [23]:
mom_rows = []
for rule in ["m1", "m2", "m3"]:
    fit = nw_ols(paper_x["Momentum"].shift(-1) * 12, rules[rule].rename("higher"), 6)
    mom_rows.append({"rule": rule, "UMD after lower": fit.params["const"], "gap (higher − lower)": fit.params["higher"],
                     "NW t": fit.tvalues["higher"]})
mom_table = pd.DataFrame(mom_rows).set_index("rule")
record("mom_gap_avg", mom_table["gap (higher − lower)"].mean(), "{:.1%}")
record("mom_t_m2", mom_table.loc["m2", "NW t"], "{:.2f}")
mom_table.round(3)

,UMD after lower,gap (higher − lower),NW t
rule,,,
m1,0.037,0.072,2.225
m2,0.025,0.084,2.613
m3,0.028,0.080,2.172


### 2.7 Exhibit 9 recomputed from the paper's assumptions

Max-Sharpe long-only portfolios from the paper's volatilities, Sharpe ratios and correlations (p.13). Expected excess return = Sharpe × volatility. Columns 2–4 add a 3% real return target, with real cash at −0.5% (column 2) or +0.5% (columns 3–4). Column 3 also adds +1% real return to every asset; column 4 applies the higher-rate Sharpe penalties. Liquid alts and Treasuries get no penalty.

In [24]:
ex9_assets = ["Public equity", "Private equity", "Treasuries", "HY bonds", "Real estate", "Liquid alts"]
vol = np.array([0.15, 0.20, 0.05, 0.10, 0.10, 0.08])
sr_norm = np.array([0.30, 0.275, 0.30, 0.275, 0.275, 0.25])
sr_pen = sr_norm + np.array([-0.06, -0.06, 0, -0.03, -0.06, 0])
corr = np.array([[1, .9, 0, .7, .5, .2], [.9, 1, 0, .7, .6, .2], [0, 0, 1, 0, .2, .2],
                 [.7, .7, 0, 1, .6, .2], [.5, .6, .2, .6, 1, .2], [.2, .2, .2, .2, .2, 1]])
cov = np.outer(vol, vol) * corr

def max_sharpe(mu_x, cash_real = None, target = None):
    n = len(mu_x)
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    if target is not None:
        cons.append({"type": "ineq", "fun": lambda w: cash_real + w @ mu_x - target})
    obj = lambda w: -(w @ mu_x) / np.sqrt(w @ cov @ w)
    res = minimize(obj, np.ones(n) / n, bounds = [(0, 1)] * n, constraints = cons, method = "SLSQP")
    return res.x

cols = {
    "(1) Unconstrained max SR":        max_sharpe(sr_norm * vol),
    "(2) Lower-rate, 3% real target":  max_sharpe(sr_norm * vol, cash_real = -0.005, target = 0.03),
    "(3) Higher-rate, same SRs":       max_sharpe(sr_norm * vol, cash_real = 0.005, target = 0.03),   # +1% real for all assets incl. cash: excess unchanged
    "(4) Higher-rate, adjusted SRs":   max_sharpe(sr_pen * vol, cash_real = 0.005, target = 0.03),
}
ex9_ours = pd.DataFrame(cols, index = ex9_assets)
ex9_paper = pd.DataFrame({"(1) Unconstrained max SR": [13, 0, 55, 9, 5, 17], "(2) Lower-rate, 3% real target": [37, 17, 18, 0, 10, 19],
                          "(3) Higher-rate, same SRs": [24, 1, 45, 2, 9, 18], "(4) Higher-rate, adjusted SRs": [26, 10, 33, 3, 0, 28]},
                         index = ex9_assets) / 100
ex9_sr = pd.Series({k: (w @ ((sr_pen if "adjusted" in k else sr_norm) * vol)) / np.sqrt(w @ cov @ w) for k, w in cols.items()})
for k, v, pv in zip(["1", "2", "3", "4"], ex9_sr, [0.45, 0.36, 0.44, 0.35]):
    record(f"ex9_sr_{k}", v, "{:.2f}", f"paper {pv}")
print("Implied Sharpe ratios (ours):", ex9_sr.round(2).tolist(), " paper: [0.45, 0.36, 0.44, 0.35]")
record("ex9_mae", (ex9_ours - ex9_paper).abs().mean().mean(), "{:.1%}", "mean absolute weight gap vs the paper")
pd.concat({"ours": ex9_ours, "paper": ex9_paper}, axis = 1).round(2)

Implied Sharpe ratios (ours): [0.45, 0.36, 0.44, 0.35]  paper: [0.45, 0.36, 0.44, 0.35]


ours                                 \
               (1) Unconstrained max SR (2) Lower-rate, 3% real target   
Public equity                      0.13                           0.37   
Private equity                     0.00                           0.17   
Treasuries                         0.55                           0.18   
HY bonds                           0.09                           0.00   
Real estate                        0.05                           0.10   
Liquid alts                        0.17                           0.18   

                                                                        \
               (3) Higher-rate, same SRs (4) Higher-rate, adjusted SRs   
Public equity                       0.24                          0.26   
Private equity                      0.02                          0.11   
Treasuries                          0.45                          0.33   
HY bonds                            0.02                          0.03   
Real estate                         0.09                          0.00   
Liquid alts                         0.18                          0.28   

                                  paper                                 \
               (1) Unconstrained max SR (2) Lower-rate, 3% real target   
Public equity                      0.13                           0.37   
Private equity                     0.00                           0.17   
Treasuries                         0.55                           0.18   
HY bonds                           0.09                           0.00   
Real estate                        0.05                           0.10   
Liquid alts                        0.17                           0.19   

                                                                        
               (3) Higher-rate, same SRs (4) Higher-rate, adjusted SRs  
Public equity                       0.24                          0.26  
Private equity                      0.01                          0.10  
Treasuries                          0.45                          0.33  
HY bonds                            0.02                          0.03  
Real estate                         0.09                          0.00  
Liquid alts                         0.18                          0.28

## <a id="s3">3 · The fiscal-deficit offset (H3)</a>

[↑ Contents](#top)

Part B. First we check the paper's channel (company earnings grow slowly after high rates), then whether a big **primary deficit** (deficit minus interest, % of GDP) offsets it. Each fiscal year's value is used from the January after it ends.

In [25]:
primary_def =\
(
    (-deficit_gdp["surplus_pct_gdp"] - interest_gdp["interest_pct_gdp"])
    .dropna()
    .pipe(lambda s: s.set_axis(pd.to_datetime((s.index.year + 1).astype(str) + "-01-01")))   # usable from next January
    .reindex(panel.index, method = "ffill")
    .rename("primary_deficit")
)
big_deficit = (primary_def > primary_def.expanding().median()).astype(float).where(primary_def.notna())

# the paper's channel: real EPS growth after higher vs lower rates (Shiller; arithmetic, annualised).
# 12 and 36 months only: Shiller interpolates quarterly earnings, so a 1-month change annualised is meaningless.
real_e = panel["real_E"].interpolate(limit_area = "inside")
eps_rows = {}
for h in [12, 36]:
    g = ((real_e.shift(-h) / real_e) ** (12 / h) - 1).loc[:PAPER_END]
    for rule in ["m1", "m2", "m3"]:
        for v, lab in [(0, "lower"), (1, "higher")]:
            eps_rows[(h, rule, lab)] = g[rules[rule] == v].mean()
eps = pd.Series(eps_rows).groupby(level = 2).mean()
record("eps_growth_lower", eps["lower"], "{:.1%}"); record("eps_growth_higher", eps["higher"], "{:.1%}")
print(f"Real EPS growth after lower rates {eps['lower']:.1%} vs after higher {eps['higher']:.1%} (paper: 11% vs 1%)")

Real EPS growth after lower rates 13.9% vs after higher -1.2% (paper: 11% vs 1%)


In [26]:
h3_x = xr["Equities"]
fwd12 = fwd_mean(h3_x, 12) * 12
eps12 = (real_e.shift(-12) / real_e - 1)
h3_df =\
(
    pd
    .DataFrame({"rates": regimes["m2"].map({1.0: "Higher rates", 0.0: "Lower rates"}),
                "deficit": big_deficit.map({1.0: "Big deficit", 0.0: "Small deficit"}),
                "next12_excess": fwd12, "next12_eps": eps12})
    .loc["1941-01":]
    .dropna(subset = ["rates", "deficit"])
)
h3_table = h3_df.groupby(["rates", "deficit"])[["next12_excess", "next12_eps"]].agg(["mean", "count"])

hi = h3_df[h3_df["rates"] == "Higher rates"].dropna(subset = ["next12_excess"])
fit_h3 = nw_ols(hi["next12_excess"], (hi["deficit"] == "Big deficit").astype(float).rename("big"), 18)
fit_h3_eps = nw_ols(hi["next12_eps"], (hi["deficit"] == "Big deficit").astype(float).rename("big"), 18)
record("h3_ret_high_big", h3_table.loc[("Higher rates", "Big deficit"), ("next12_excess", "mean")], "{:.1%}")
record("h3_ret_high_small", h3_table.loc[("Higher rates", "Small deficit"), ("next12_excess", "mean")], "{:.1%}")
record("h3_ret_low_big", h3_table.loc[("Lower rates", "Big deficit"), ("next12_excess", "mean")], "{:.1%}")
record("h3_ret_low_small", h3_table.loc[("Lower rates", "Small deficit"), ("next12_excess", "mean")], "{:.1%}")
record("h3_diff_high", fit_h3.params["big"], "{:.1%}"); record("h3_t_high", fit_h3.tvalues["big"], "{:.2f}")
record("h3_eps_diff_high", fit_h3_eps.params["big"], "{:.1%}"); record("h3_eps_t_high", fit_h3_eps.tvalues["big"], "{:.2f}")
print(f"Within higher-rate months (M2), big vs small deficit: next-12M return above cash {fit_h3.params['big']:+.1%} "
      f"(NW t = {fit_h3.tvalues['big']:.2f}); real EPS growth {fit_h3_eps.params['big']:+.1%} (t = {fit_h3_eps.tvalues['big']:.2f})")
h3_table.round(3)

Within higher-rate months (M2), big vs small deficit: next-12M return above cash -4.7% (NW t = -1.30); real EPS growth +1.2% (t = 0.31)


next12_excess       next12_eps      
                                    mean count       mean count
rates        deficit                                           
Higher rates Big deficit           0.054   295     -0.001   292
             Small deficit         0.101   273     -0.013   273
Lower rates  Big deficit           0.109   314      0.268   314
             Small deficit         0.078   135      0.103   135

In [27]:
fig04_df =\
(
    h3_table[("next12_excess", "mean")].rename("ret").reset_index()
    .assign(rates = lambda d: pd.Categorical(d["rates"], ["Lower rates", "Higher rates"]),
            deficit = lambda d: pd.Categorical(d["deficit"], ["Small deficit", "Big deficit"]))
)
fig04 =\
(
    ggplot(fig04_df, aes(x = "rates", y = "ret", fill = "deficit"))
    + geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.75)
    + geom_text(aes(label = "ret"), label_format = ".1%", position = position_dodge(width = 0.8), vjust = -0.6, size = 7)
    + scale_fill_manual(values = [PALETTE[4], PALETTE[1]])
    + scale_y_continuous(format = ".0%")
    + labs(title = "H3: does a big deficit offset high rates?",
           subtitle = "Stocks' return above cash over the next 12 months, by rate regime (M2) and primary deficit, 1941 – 2026",
           caption = "Source: Ken French, FRED; Group 6 calculations",
           x = "", y = "Return above cash, next 12 months", fill = "")
    + theme(text = element_text(size = 14), legend_position = "top")
    + ggsize(900, 450)
)
save_fig(fig04, "fig04_h3_deficit")
fig04

## <a id="s4">4 · Out-of-sample tests</a>

[↑ Contents](#top)

Does the pattern forecast, or only describe the past? (Prof Liu points 2, 3, 5.)

### 4.1 Fit on 1926–1995, test on 1996–2026

We fit the predictive regression (next month's equity return above cash on today's T-bill rate) on Jul 1926 – Dec 1995 only, then forecast Jan 1996 – Sep 2026. Out-of-sample R² compares the squared forecast errors with those of the 1926–95 average return (Campbell & Thompson 2008): above 0 means the rate helps.

In [28]:
d41 = pd.concat([xr["Equities"].shift(-1).rename("y"), rate.rename("tbill"), (100 / SHILLER["CAPE"]).rename("ey")], axis = 1).dropna()
train, test = d41.loc[:"1995-12"], d41.loc["1996-01":]
fit_tr = sm.OLS(train["y"], sm.add_constant(train["tbill"])).fit()
pred = fit_tr.params["const"] + fit_tr.params["tbill"] * test["tbill"]
r2_oos = 1 - ((test["y"] - pred) ** 2).sum() / ((test["y"] - train["y"].mean()) ** 2).sum()
fit_te = nw_ols(test["y"], test[["tbill"]], 12)
record("oos_r2", r2_oos, "{:.2%}")
record("oos_slope_train_t", fit_tr.tvalues["tbill"], "{:.2f}")
record("oos_slope_test", fit_te.params["tbill"] * 12, "{:.2%}"); record("oos_slope_test_t", fit_te.tvalues["tbill"], "{:.2f}")

# Exhibit-3-style equity gap in the later period (M2 only: the other rules use the full sample or the future)
late = xr["Equities"].loc["1996-01":]
gap_late = exhibit3(late, regimes[["m2"]].loc[late.index], [12])
record("late_eq_lower_12m", gap_late[(12, "m2", "Lower")], "{:.1%}"); record("late_eq_higher_12m", gap_late[(12, "m2", "Higher")], "{:.1%}")
print(f"Train slope t = {fit_tr.tvalues['tbill']:.2f}; test slope = {fit_te.params['tbill']*12:+.2%} per 1pp (NW t = {fit_te.tvalues['tbill']:.2f}); "
      f"out-of-sample R² = {r2_oos:.2%}")
print(f"1996–2026, M2: next-12M equity return above cash after lower {gap_late[(12,'m2','Lower')]:.1%} vs higher {gap_late[(12,'m2','Higher')]:.1%}")

Train slope t = -1.82; test slope = -1.05% per 1pp (NW t = -0.79); out-of-sample R² = 0.15%
1996–2026, M2: next-12M equity return above cash after lower 10.0% vs higher 7.6%


### 4.2 The recent years on their own (2024–2026)

Stocks' calendar-year return above cash, the share of higher-rate months (M2), and the paper-sample average for higher-rate months.

In [29]:
yr = (1 + xr["Equities"]).groupby(xr.index.year).prod() - 1
recent = pd.DataFrame({"equity return above cash": yr.loc[2022:2026],
                       "share of months higher-rate (M2)": regimes["m2"].groupby(regimes.index.year).mean().loc[2022:2026]})
hi_avg = (fwd_mean(xr["Equities"].loc[:PAPER_END], 12) * 12)[rules["m2"] == 1].mean()
recent["paper-sample average after higher rates"] = hi_avg
for y in [2024, 2025, 2026]:
    record(f"recent_eq_{y}", recent.loc[y, "equity return above cash"], "{:.1%}", "2026 = Jan–Sep" if y == 2026 else "")
record("recent_hi_avg", hi_avg, "{:.1%}")
recent.round(3)

,equity return above cash,share of months higher-rate (M2),paper-sample average after higher rates
Date,,,
2022,-0.210,0.750,0.07
2023,0.208,1.000,0.07
2024,0.189,1.000,0.07
2025,0.128,0.833,0.07
2026,0.090,0.111,0.07


### 4.3 Robustness: a simpler rule and valuation (CAPE)

- **Simpler rule:** higher-rate if the T-bill rate is above its own trailing 5-year **average**.
- **CAPE:** the paper's two-factor regression on the T-bill rate and 1/CAPE (paper: t = −3.2 and +3.9, p.7).

In [30]:
simple = ((rate > rate.shift(1).rolling(60).mean()) & (rate >= 0.5)).astype(float).where(rate.shift(1).rolling(60).count() == 60)
simple_gap = exhibit3(xr["Equities"].loc[:PAPER_END], pd.DataFrame({"simple": simple}).loc[:PAPER_END], [1, 12])
record("simple_eq_lower_12m", simple_gap[(12, "simple", "Lower")], "{:.1%}"); record("simple_eq_higher_12m", simple_gap[(12, "simple", "Higher")], "{:.1%}")

cape_rows = []
for label, end in [("Jul 1926 – Dec 2023", PAPER_END), ("Jul 1926 – Sep 2026", None)]:
    d = d41.loc[:end]
    f_ols = sm.OLS(d["y"], sm.add_constant(d[["tbill", "ey"]])).fit()
    f_nw = nw_ols(d["y"], d[["tbill", "ey"]], 12)
    cape_rows.append({"sample": label, "T-bill OLS t": f_ols.tvalues["tbill"], "1/CAPE OLS t": f_ols.tvalues["ey"],
                      "T-bill NW t": f_nw.tvalues["tbill"], "1/CAPE NW t": f_nw.tvalues["ey"]})
cape_table = pd.DataFrame(cape_rows).set_index("sample")
record("cape_t_tbill", cape_table.iloc[0]["T-bill OLS t"], "{:.2f}"); record("cape_t_ey", cape_table.iloc[0]["1/CAPE OLS t"], "{:.2f}")
record("cape_latest", SHILLER["CAPE"].dropna().iloc[-1], "{:.1f}")
print("Simpler rule, next 12M equity return above cash:", {k: f"{v:.1%}" for k, v in simple_gap.items()})
cape_table.round(2)

Simpler rule, next 12M equity return above cash: {(1, 'simple', 'Higher'): '4.8%', (1, 'simple', 'Lower'): '11.5%', (12, 'simple', 'Higher'): '6.8%', (12, 'simple', 'Lower'): '10.6%'}


,T-bill OLS t,1/CAPE OLS t,T-bill NW t,1/CAPE NW t
sample,,,,
Jul 1926 – Dec 2023,-3.15,4.03,-2.68,2.73
Jul 1926 – Sep 2026,-2.99,3.78,-2.54,2.56


### 4.4 Why 2024–26 broke the pattern: concentration

Cap-weighted S&P 500 (SPY) vs equal-weight (RSP). If a few very large companies drove the gains, SPY beats RSP by a wide margin.

In [31]:
conc = (ETF[["SPY", "RSP"]].resample("YE").last().pct_change().loc["2022":])
conc.index = conc.index.year
conc["SPY − RSP"] = conc["SPY"] - conc["RSP"]
for y in [2023, 2024, 2025]:
    record(f"conc_gap_{y}", conc.loc[y, "SPY − RSP"], "{:+.1%}")
conc.round(3)

Ticker,SPY,RSP,SPY − RSP
Date,,,
2022,-0.182,-0.116,-0.066
2023,0.262,0.137,0.125
2024,0.249,0.128,0.121
2025,0.177,0.112,0.065
2026,0.117,0.153,-0.036


## <a id="s5">5 · What should an investor hold?</a>

[↑ Contents](#top)

Walkthrough step 5, on the **longest sample the data allows**: monthly, Feb 1960 → Sep 2026 (World Bank gold starts Jan 1960). Four assets: **equity** (Ken French market), **bonds** (10-year Treasury return, §1.8), **gold** (World Bank), **momentum** (RF + UMD, the cash-plus stand-in). Cash = Ken French RF. All conditioning uses information known at the end of the previous month.

### 5.0 Sample, states and performance measures

In [32]:
A_RISK = 3          # risk aversion for w* and the certainty equivalent (Lecture 1–2); results also shown for 2 and 5

def perf(r, rf):
    """Annualised performance of a monthly return series r (rf = monthly cash return)."""
    r, rf = r.dropna(), rf.reindex(r.dropna().index)
    x = r - rf
    wealth = (1 + r).cumprod()
    return pd.Series({
        "ann_return": (1 + r).prod() ** (12 / len(r)) - 1,
        "volatility": r.std() * np.sqrt(12),
        "sharpe":     x.mean() / x.std() * np.sqrt(12),
        "max_drawdown": (wealth / wealth.cummax() - 1).min(),
        "cvar5":      -r[r <= r.quantile(0.05)].mean(),          # average loss in the worst 5% of months
        "ce":         x.mean() * 12 - 0.5 * A_RISK * x.var() * 12,  # certainty equivalent above cash
        "months":     len(r),
    })

s5_assets =\
(
    panel[["equity", "bond10", "gold", "mom_cash", "rf_m"]]
    .loc["1960-02":]
    .dropna()
)
s5_assets.columns = ["Equity", "Bonds", "Gold", "Momentum", "Cash"]

# rate states known at the end of last month
chg12 = rate.diff(12)
state =\
(
    pd
    .DataFrame({"high": regimes["m2"], "chg12": chg12})
    .assign(direction = lambda d: np.select([d["chg12"] > 0.5, d["chg12"] < -0.5], ["rising", "falling"], "steady"),
            state = lambda d: np.where(d["direction"] != "steady", d["direction"],
                                       np.where(d["high"] == 1, "high & steady", "low & steady")))
    .shift(1)
    .reindex(s5_assets.index)
)
state["regime"] = np.select([state["high"] == 1, state["high"] == 0], ["higher-rate", "lower-rate"], None)
print(f"Sample {s5_assets.index[0]:%b %Y} – {s5_assets.index[-1]:%b %Y}, {len(s5_assets)} months")
state["state"].value_counts().to_frame("months")

Sample Feb 1960 – Sep 2026, 800 months


,months
state,
falling,264
rising,262
low & steady,157
high & steady,117


### 5.1 Which assets still diversify stocks when rates are high, rising or falling?

Correlation of each asset's monthly return with equity, and its average return above cash, by rate state. A diversifier has low or negative correlation with equity in that state.

In [33]:
rows = []
for label, mask in [("all months", slice(None))] + \
                   [(g, state["regime"] == g) for g in ["higher-rate", "lower-rate"]] + \
                   [(g, state["state"] == g) for g in ["rising", "high & steady", "low & steady", "falling"]]:
    sub = s5_assets.loc[mask]
    for a in ["Bonds", "Gold", "Momentum"]:
        rows.append({"state": label, "asset": a, "months": len(sub),
                     "corr_with_equity": sub[a].corr(sub["Equity"]),
                     "excess_return": (sub[a] - sub["Cash"]).mean() * 12})
diversifiers = pd.DataFrame(rows).pivot(index = "state", columns = "asset", values = ["corr_with_equity", "excess_return"])
for a in ["Bonds", "Gold", "Momentum"]:
    record(f"s5_corr_{a.lower()}_high", diversifiers.loc["higher-rate", ("corr_with_equity", a)], "{:.2f}")
    record(f"s5_corr_{a.lower()}_rising", diversifiers.loc["rising", ("corr_with_equity", a)], "{:.2f}")
diversifiers.round(3)

corr_with_equity                 excess_return                
asset                    Bonds   Gold Momentum         Bonds   Gold Momentum
state                                                                       
all months               0.128 -0.030   -0.167         0.016  0.042    0.076
falling                  0.062 -0.046   -0.343         0.036  0.060    0.051
high & steady            0.161 -0.142   -0.216         0.020  0.008    0.131
higher-rate              0.219  0.001   -0.084         0.004  0.039    0.105
low & steady            -0.084 -0.063   -0.111         0.017  0.035    0.060
lower-rate               0.028 -0.074   -0.252         0.031  0.046    0.040
rising                   0.285  0.021    0.052        -0.006  0.042    0.086

### 5.2 Mean-variance stock share by regime

For one risky asset plus cash, the optimal risky share is **w\* = (E[R] − r_f) / (A σ²)** (Lecture 2). We estimate it from equity's excess return in higher- and lower-rate months. If the premium shrinks but volatility also falls, w\* falls by less (the H2 argument).

In [34]:
rows = []
for g in ["higher-rate", "lower-rate"]:
    x = (s5_assets["Equity"] - s5_assets["Cash"])[state["regime"] == g]
    mu, var = x.mean() * 12, x.var() * 12
    rows.append({"regime": g, "months": len(x), "excess_return": mu, "volatility": np.sqrt(var),
                 "sharpe": mu / np.sqrt(var), **{f"w* (A={a})": mu / (a * var) for a in (2, 3, 5)}})
mv_share = pd.DataFrame(rows).set_index("regime")
record("s5_wstar_high_A3", mv_share.loc["higher-rate", "w* (A=3)"], "{:.0%}")
record("s5_wstar_low_A3",  mv_share.loc["lower-rate", "w* (A=3)"],  "{:.0%}")
record("s5_eq_sharpe_high", mv_share.loc["higher-rate", "sharpe"], "{:.2f}")
record("s5_eq_sharpe_low",  mv_share.loc["lower-rate", "sharpe"],  "{:.2f}")
mv_share.round(3)

,months,excess_return,volatility,sharpe,w* (A=2),w* (A=3),w* (A=5)
regime,,,,,,,
higher-rate,440,0.046,0.152,0.302,0.996,0.664,0.398
lower-rate,360,0.105,0.156,0.672,2.156,1.437,0.862


### 5.3 Fixed mixes in rising, steady and falling rate periods

100% stocks, 70/30, 60/40, 50/50 (stocks/bonds) and **1/N** (equal weights in equity, bonds, gold, momentum), rebalanced monthly. Maximum drawdown is measured on the months of each state joined together, so read it as indicative.

In [35]:
mixes = {"100/0":  {"Equity": 1.0},
         "70/30":  {"Equity": 0.7, "Bonds": 0.3},
         "60/40":  {"Equity": 0.6, "Bonds": 0.4},
         "50/50":  {"Equity": 0.5, "Bonds": 0.5},
         "1/N":    {"Equity": 0.25, "Bonds": 0.25, "Gold": 0.25, "Momentum": 0.25}}
mix_returns = pd.DataFrame({k: sum(s5_assets[a] * w for a, w in v.items()) for k, v in mixes.items()})

rows = []
for label, mask in [("all months", pd.Series(True, index = state.index))] + \
                   [(g, state["regime"] == g) for g in ["higher-rate", "lower-rate"]] + \
                   [(g, state["state"] == g) for g in ["rising", "high & steady", "low & steady", "falling"]]:
    for k in mixes:
        rows.append({"state": label, "mix": k, **perf(mix_returns.loc[mask, k], s5_assets.loc[mask, "Cash"])})
mix_table = pd.DataFrame(rows).set_index(["state", "mix"])

for st, key in [("higher-rate", "high"), ("rising", "rising"), ("falling", "falling")]:
    for k, kk in [("60/40", "6040"), ("50/50", "5050"), ("1/N", "1n")]:
        record(f"s5_sharpe_{kk}_{key}", mix_table.loc[(st, k), "sharpe"], "{:.2f}")
        record(f"s5_cvar_{kk}_{key}", mix_table.loc[(st, k), "cvar5"], "{:.1%}")
mix_table[["ann_return", "volatility", "sharpe", "max_drawdown", "cvar5", "months"]].round(3)

ann_return  volatility  sharpe  max_drawdown  cvar5  \
state         mix                                                          
all months    100/0       0.108       0.153   0.471        -0.503  0.096   
              70/30       0.096       0.112   0.495        -0.349  0.067   
              60/40       0.091       0.099   0.502        -0.306  0.059   
              50/50       0.086       0.087   0.505        -0.261  0.050   
              1/N         0.096       0.067   0.769        -0.153  0.034   
higher-rate   100/0       0.093       0.151   0.302        -0.520  0.093   
              70/30       0.085       0.112   0.296        -0.374  0.066   
              60/40       0.082       0.099   0.290        -0.318  0.058   
              50/50       0.079       0.088   0.279        -0.261  0.050   
              1/N         0.106       0.070   0.681        -0.142  0.035   
lower-rate    100/0       0.127       0.156   0.672        -0.459  0.100   
              70/30       0.109       0.112   0.742        -0.322  0.068   
              60/40       0.102       0.098   0.769        -0.270  0.058   
              50/50       0.096       0.086   0.794        -0.215  0.049   
              1/N         0.084       0.062   0.895        -0.153  0.033   
rising        100/0       0.048       0.152  -0.001        -0.521  0.102   
              70/30       0.051       0.114  -0.018        -0.400  0.074   
              60/40       0.052       0.102  -0.026        -0.354  0.065   
              50/50       0.052       0.092  -0.036        -0.305  0.057   
              1/N         0.089       0.079   0.378        -0.178  0.043   
high & steady 100/0       0.117       0.162   0.480        -0.286  0.093   
              70/30       0.105       0.116   0.518        -0.208  0.065   
              60/40       0.101       0.102   0.535        -0.180  0.056   
              50/50       0.096       0.088   0.555        -0.152  0.048   
              1/N         0.110       0.052   1.137        -0.071  0.025   
low & steady  100/0       0.109       0.132   0.749        -0.234  0.081   
              70/30       0.087       0.093   0.802        -0.174  0.055   
              60/40       0.079       0.081   0.818        -0.153  0.047   
              50/50       0.072       0.070   0.826        -0.135  0.041   
              1/N         0.067       0.053   1.012        -0.076  0.026   
falling       100/0       0.167       0.161   0.781        -0.503  0.094   
              70/30       0.143       0.116   0.853        -0.345  0.062   
              60/40       0.134       0.103   0.878        -0.284  0.051   
              50/50       0.126       0.091   0.899        -0.218  0.042   
              1/N         0.113       0.066   1.033        -0.153  0.030   

                     months  
state         mix            
all months    100/0   800.0  
              70/30   800.0  
              60/40   800.0  
              50/50   800.0  
              1/N     800.0  
higher-rate   100/0   440.0  
              70/30   440.0  
              60/40   440.0  
              50/50   440.0  
              1/N     440.0  
lower-rate    100/0   360.0  
              70/30   360.0  
              60/40   360.0  
              50/50   360.0  
              1/N     360.0  
rising        100/0   262.0  
              70/30   262.0  
              60/40   262.0  
              50/50   262.0  
              1/N     262.0  
high & steady 100/0   117.0  
              70/30   117.0  
              60/40   117.0  
              50/50   117.0  
              1/N     117.0  
low & steady  100/0   157.0  
              70/30   157.0  
              60/40   157.0  
              50/50   157.0  
              1/N     157.0  
falling       100/0   264.0  
              70/30   264.0  
              60/40   264.0  
              50/50   264.0  
              1/N     264.0

### 5.4 Good, middle and bad conditions: the best stock share

We split history into thirds by three variables known at the end of the previous month, then test stock/bond mixes from 0% to 100% stocks in 10% steps within each third (Prof Liu points 8–9):
- **T-bill rate:** low third = good (the paper's direction).
- **Valuation gap:** excess CAPE yield = 1/CAPE − real 10-year yield (10Y yield − last 12 months' CPI inflation). High = stocks cheap = good.
- **Primary deficit** (deficit minus interest, % of GDP; fiscal-year value used from January of the following year): big = good (H3).

The tercile cut-offs use the whole sample (descriptive, not a trading rule). The best share is reported by Sharpe and by certainty equivalent (A = 3).

In [36]:
infl12 = CPI["CPI"].pct_change(12) * 100
ecy = (100 / SHILLER["CAPE"] - (y10 - infl12)).rename("valuation_gap")
primary_def =\
(
    (-deficit_gdp["surplus_pct_gdp"] - interest_gdp["interest_pct_gdp"])
    .dropna()
    .pipe(lambda s: s.set_axis(pd.to_datetime((s.index.year + 1).astype(str) + "-01-01")))   # usable from next January
    .reindex(panel.index, method = "ffill")
    .rename("primary_deficit")
)
conditions =\
(
    pd
    .concat([rate.rename("tbill"), ecy, primary_def], axis = 1)
    .shift(1)
    .reindex(s5_assets.index)
)
good_when_high = {"tbill": False, "valuation_gap": True, "primary_deficit": True}

def terciles(s, good_high):
    t = pd.qcut(s, 3, labels = ["bottom", "middle", "top"])
    return t.map({"bottom": "bad" if good_high else "good", "middle": "middle", "top": "good" if good_high else "bad"})

rows = []
for var, good_high in good_when_high.items():
    lab = terciles(conditions[var], good_high)
    for third in ["good", "middle", "bad"]:
        m = lab == third
        for w in np.round(np.arange(0, 1.01, 0.1), 1):
            r = w * s5_assets.loc[m, "Equity"] + (1 - w) * s5_assets.loc[m, "Bonds"]
            rows.append({"variable": var, "third": third, "stock_share": w, **perf(r, s5_assets.loc[m, "Cash"])})
share_grid = pd.DataFrame(rows)

best_share =\
(
    share_grid
    .groupby(["variable", "third"])
    .apply(lambda g: pd.Series({"best_by_sharpe": g.loc[g["sharpe"].idxmax(), "stock_share"],
                                "best_by_ce":     g.loc[g["ce"].idxmax(), "stock_share"],
                                "sharpe_at_best": g["sharpe"].max(),
                                "months":         int(g["months"].iloc[0])}))
    .reindex(pd.MultiIndex.from_product([list(good_when_high), ["good", "middle", "bad"]]))
)

# where is today? latest available value of each variable, placed in the sample's thirds
latest = pd.concat([rate.rename("tbill"), ecy, primary_def], axis = 1).ffill().iloc[-1]
today = {}
for var, good_high in good_when_high.items():
    cuts = conditions[var].quantile([1 / 3, 2 / 3]).values
    pos = "bottom" if latest[var] <= cuts[0] else ("middle" if latest[var] <= cuts[1] else "top")
    today[var] = {"bottom": "bad" if good_high else "good", "middle": "middle", "top": "good" if good_high else "bad"}[pos]
    record(f"s5_today_{var}", latest[var], "{:.2f}", f"latest value; today's third = {today[var]}")
    record(f"s5_best_share_{var}_today", best_share.loc[(var, today[var]), "best_by_sharpe"], "{:.0%}",
           f"best stock share by Sharpe in today's third ({today[var]})")
    record(f"s5_best_share_{var}_middle", best_share.loc[(var, "middle"), "best_by_sharpe"], "{:.0%}")
print("Today's third:", today)
best_share

Today's third: {'tbill': 'middle', 'valuation_gap': 'bad', 'primary_deficit': 'good'}


/var/folders/gv/7pyqtc6n40bgn3f0ypm9j6h00000gn/T/ipykernel_83661/4118193856.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"best_by_sharpe": g.loc[g["sharpe"].idxmax(), "stock_share"],


best_by_sharpe  best_by_ce  sharpe_at_best  months
tbill           good               0.3         0.9        0.729063   267.0
                middle             1.0         1.0        0.685209   267.0
                bad                0.3         0.3        0.308722   266.0
valuation_gap   good               0.8         0.5        0.276069   267.0
                middle             0.7         1.0        0.852113   265.0
                bad                0.3         0.6        0.611786   267.0
primary_deficit good               0.6         1.0        0.562855   260.0
                middle             1.0         1.0        0.670314   264.0
                bad                0.2         0.4        0.418268   276.0

In [37]:
fig05 =\
(
    ggplot(share_grid.assign(third = lambda d: pd.Categorical(d["third"], ["good", "middle", "bad"])),
           aes(x = "stock_share", y = "sharpe", color = "third"))
    + geom_line(size = 1.4)
    + geom_point(size = 2.5)
    + facet_wrap("variable", ncol = 3)
    + scale_color_manual(values = PALETTE[:3])
    + scale_x_continuous(format = ".0%")
    + labs(title = "Best stock share in good, middle and bad conditions",
           subtitle = f"Stock/bond mixes, monthly {s5_assets.index[0]:%b %Y} – {s5_assets.index[-1]:%b %Y}",
           caption = "Source: Ken French, FRED, Shiller, World Bank; Group 6 calculations",
           x = "Share in stocks", y = "Sharpe ratio", color = "")
    + theme(text = element_text(size = 14), legend_position = "top")
    + ggsize(1100, 450)
)
save_fig(fig05, "fig05_best_share_terciles")
fig05

### 5.5 The recommended stock share today (the X in the central message)

Four pieces of evidence, each answering "how much in stocks now?":
- the mean-variance share for higher-rate months (A = 3);
- the best stock share by Sharpe in today's third for the T-bill rate, for valuation, and for the deficit.

We take their average, rounded to the nearest 10%.

In [38]:
evidence = pd.Series({"Mean-variance w* (higher-rate, A = 3)": mv_share.loc["higher-rate", "w* (A=3)"],
                      **{f"Best share, today's {v} third ({today[v]})": best_share.loc[(v, today[v]), "best_by_sharpe"]
                         for v in good_when_high}})
x_share = round(evidence.clip(0, 1).mean() * 10) / 10
record("x_stock_share", x_share, "{:.0%}", "recommended stock share today (central message)")
print(f"Recommended stock share today: {x_share:.0%}")
evidence.round(2).to_frame("stock share")

Recommended stock share today: 60%


,stock share
"Mean-variance w* (higher-rate, A = 3)",0.66
"Best share, today's tbill third (middle)",1.00
"Best share, today's valuation_gap third (bad)",0.30
"Best share, today's primary_deficit third (good)",0.60


## <a id="s6">6 · Backtests & simulation</a>

[↑ Contents](#top)

Walkthrough step 6 on **investable ETFs**: SPY (stocks), IEF (Treasuries), GLD (gold), PDP (momentum), plus PSP, HYG and VNQ for the paper's Exhibit 9 portfolios (S1). Four strategies are compared with one benchmark (B). Monthly, May 2007 → Sep 2026, split **chronologically 70/30**: everything is fitted or chosen on the first 70% (in-sample) and judged only on the last 30% (out-of-sample), with no shuffling (Prof Liu point 5). Rebalanced monthly; trading costs of 10 bps per unit of turnover. The Monte Carlo of “rates stay high” vs “rates fall” is still to do.

### 6.1 Monthly ETF returns and the 70/30 split

In [39]:
COST_BPS = 10
etf_m =\
(
    ETF[["SPY", "IEF", "GLD", "PDP", "PSP", "HYG", "VNQ"]]
    .resample("MS").last()                 # month-start labels, to line up with the panel
    .pct_change()
    .loc["2007-05":]
    .dropna()
)
cash_m = panel["rf_m"].reindex(etf_m.index)
etf_m, cash_m = etf_m.loc[cash_m.dropna().index], cash_m.dropna()

n_is = int(round(len(etf_m) * 0.7))
IS, OOS = etf_m.index[:n_is], etf_m.index[n_is:]
sig = regimes["regime_m2_lag"].reindex(etf_m.index)

# big primary deficit (above its median to date), known at the end of last month
big_def = (primary_def > primary_def.expanding().median()).shift(1).reindex(etf_m.index)

print(f"In-sample  {IS[0]:%b %Y} – {IS[-1]:%b %Y}: {len(IS)} months, {int(sig[IS].sum())} higher-rate")
print(f"Out-of-sample {OOS[0]:%b %Y} – {OOS[-1]:%b %Y}: {len(OOS)} months, {int(sig[OOS].sum())} higher-rate")
record("s6_is_start", IS[0].year, "{:.0f}"); record("s6_oos_start", OOS[0].year + OOS[0].month / 100, "{:.2f}")

In-sample  May 2007 – Nov 2020: 163 months, 49 higher-rate
Out-of-sample Dec 2020 – Sep 2026: 70 months, 43 higher-rate


2020.12

### 6.2 The benchmark and the four strategies

| Code | Rule ("high" = previous month's M2 label is higher-rate) |
|------|------|
| **B** | **Benchmark:** equal weights in SPY, IEF, GLD, PDP, bought once and **never rebalanced**. It restarts at 25% each at the start of each period (in-sample, out-of-sample), so both periods start from the same line. |
| **S1** | **The paper's portfolios (Exhibit 9), the reference:** low = 37 SPY / 17 PSP / 18 IEF / 10 VNQ / 19 PDP; high = 26 SPY / 10 PSP / 33 IEF / 3 HYG / 28 PDP (rescaled to 100%) |
| **S2** | Rate rule (the paper's direction): 70/30 SPY/IEF when low; 50/50 when high |
| **S3** | Rate + deficit (H3): like S2, but 60/40 when rates are high **and** the primary deficit is big |
| **S4** | Ridge mean-variance on SPY, IEF, GLD, PDP, estimated **in-sample** for each regime (w = (λΣ + ηI)⁻¹μ, long-only, fully invested; Lecture 5); η chosen in-sample |

S1–S4 rebalance monthly and pay 10 bps per unit of turnover; B never trades after its first purchase. Alpha and beta come from regressing each strategy's excess return on the benchmark's (as in Assignment 1), with Newey-West t-statistics.

In [40]:
TICK = list(etf_m.columns)

def w(**kw):
    s = pd.Series(kw, dtype = float).reindex(TICK).fillna(0.0)
    return s / s.sum()

def by_regime(low, high, alt = None, alt_mask = None):
    out = pd.DataFrame([high if h == 1 else low for h in sig], index = etf_m.index)
    if alt is not None:
        out.loc[alt_mask] = alt.values
    return out

def backtest(weights, rets = etf_m, cost_bps = COST_BPS):
    """Monthly rebalance to target weights; cost on turnover from drifted weights."""
    r_gross, turnover, held = [], [], None
    for t in rets.index:
        target = weights.loc[t]
        turn = target.abs().sum() if held is None else (target - held).abs().sum()
        r = float((target * rets.loc[t]).sum())
        held = target * (1 + rets.loc[t]) / (1 + r)
        r_gross.append(r - turn * cost_bps / 1e4)
        turnover.append(turn)
    return pd.Series(r_gross, index = rets.index), pd.Series(turnover, index = rets.index)

def buy_and_hold(assets, idx_):
    """Equal weights bought at the start of idx_ and never rebalanced (no trading cost after purchase)."""
    wealth = (1 + etf_m.loc[idx_, assets]).cumprod().mean(axis = 1)
    return wealth.pct_change().fillna(wealth.iloc[0] - 1)

B_ASSETS = ["SPY", "IEF", "GLD", "PDP"]
bench = pd.concat([buy_and_hold(B_ASSETS, IS), buy_and_hold(B_ASSETS, OOS)])   # restarts at 25% each in Dec 2020

strategies = {
    "S1 Paper (Exhibit 9)": by_regime(w(SPY = 37, PSP = 17, IEF = 18, VNQ = 10, PDP = 19),
                                      w(SPY = 26, PSP = 10, IEF = 33, HYG = 3, PDP = 28)),
    "S2 Rate rule": by_regime(w(SPY = 70, IEF = 30), w(SPY = 50, IEF = 50)),
    "S3 Rate + deficit": by_regime(w(SPY = 70, IEF = 30), w(SPY = 50, IEF = 50),
                                   alt = w(SPY = 60, IEF = 40), alt_mask = (sig == 1) & (big_def == 1)),
}

**S4 · ridge mean-variance by regime, fitted in-sample only.** For each regime we estimate μ (excess returns) and Σ from in-sample months, solve w = (λΣ + ηI)⁻¹μ with λ = A = 3, drop negative weights and rescale to 100%. η is picked from a grid by in-sample Sharpe. The out-of-sample period never touches the estimate.

In [41]:
S4_ASSETS = ["SPY", "IEF", "GLD", "PDP"]
xs = etf_m[S4_ASSETS].sub(cash_m, axis = 0)

def ridge_weights(mask, eta, lam = A_RISK):
    mu, S = xs.loc[mask].mean() * 12, xs.loc[mask].cov() * 12
    raw = np.linalg.solve(lam * S.values + eta * np.eye(len(mu)), mu.values)
    raw = np.clip(raw, 0, None)
    return w(**dict(zip(S4_ASSETS, raw))) if raw.sum() > 0 else w(**{a: 1 for a in S4_ASSETS})

def s4_weights(eta):
    w_lo = ridge_weights(IS[sig[IS] == 0], eta)
    w_hi = ridge_weights(IS[sig[IS] == 1], eta) if (sig[IS] == 1).sum() >= 24 else w_lo   # need ≥ 2 years of data
    return by_regime(w_lo, w_hi), w_lo, w_hi

eta_grid = [0.01, 0.05, 0.1, 0.5, 1.0]
is_sharpe = {eta: perf(backtest(s4_weights(eta)[0])[0][IS], cash_m[IS])["sharpe"] for eta in eta_grid}
best_eta = max(is_sharpe, key = is_sharpe.get)
strategies["S4 Ridge MV"], s4_lo, s4_hi = s4_weights(best_eta)
record("s6_s4_eta", best_eta, "{:g}", "ridge eta chosen in-sample")
print(f"η chosen in-sample: {best_eta} · higher-rate in-sample months: {int((sig[IS] == 1).sum())}")
pd.DataFrame({"low-rate weights": s4_lo, "high-rate weights": s4_hi}).loc[S4_ASSETS].round(3)

η chosen in-sample: 0.01 · higher-rate in-sample months: 49


,low-rate weights,high-rate weights
SPY,0.377,0.054
IEF,0.585,0.485
GLD,0.000,0.353
PDP,0.038,0.107


### 6.3 In-sample vs out-of-sample results

In [42]:
import statsmodels.api as sm

def alpha_beta(r, bench_r, rf):
    """Regress strategy excess return on benchmark excess return (monthly), Newey-West (3 lags)."""
    y, x = (r - rf).dropna(), (bench_r - rf).reindex((r - rf).dropna().index)
    fit = sm.OLS(y, sm.add_constant(x)).fit(cov_type = "HAC", cov_kwds = {"maxlags": 3})
    return {"alpha": fit.params.iloc[0] * 12, "alpha_t": fit.tvalues.iloc[0], "beta": fit.params.iloc[1]}

order = ["B Benchmark (1/N held)", "S1 Paper (Exhibit 9)", "S2 Rate rule", "S3 Rate + deficit", "S4 Ridge MV"]
returns, turnovers = {"B Benchmark (1/N held)": bench}, {"B Benchmark (1/N held)": pd.Series(0.0, index = etf_m.index)}
for name in order[1:]:
    returns[name], turnovers[name] = backtest(strategies[name])

rows = []
for period, idx_ in [("In-sample", IS), ("Out-of-sample", OOS)]:
    for name in order:
        r = returns[name][idx_]
        p = perf(r, cash_m[idx_])
        ab = alpha_beta(r, returns[order[0]][idx_], cash_m[idx_]) if name != order[0] else \
             {"alpha": np.nan, "alpha_t": np.nan, "beta": np.nan}
        rows.append({"period": period, "strategy": name,
                     "Alpha (annualised)": ab["alpha"], "Alpha t-stat (NW)": ab["alpha_t"], "Beta": ab["beta"],
                     "Annual return": p["ann_return"], "Annual variance": r.var() * 12,
                     "Annual Sharpe": p["sharpe"], "CVaR 5% (monthly)": p["cvar5"],
                     "Max drawdown": p["max_drawdown"], "Turnover p.a.": turnovers[name][idx_].mean() * 12})
summary6 = pd.DataFrame(rows).set_index(["period", "strategy"]).T.rename_axis("Metric")

codes = {name: name.split()[0].lower() for name in order}
for (period, name), col in summary6.items():
    tag = "is" if period == "In-sample" else "oos"
    for metric, key, fmt in [("Alpha (annualised)", "alpha", "{:.2%}"), ("Alpha t-stat (NW)", "alpha_t", "{:.2f}"),
                             ("Beta", "beta", "{:.2f}"), ("Annual return", "ret", "{:.1%}"),
                             ("Annual variance", "var", "{:.4f}"), ("Annual Sharpe", "sharpe", "{:.2f}"),
                             ("CVaR 5% (monthly)", "cvar5", "{:.1%}"), ("Max drawdown", "mdd", "{:.1%}")]:
        if pd.notna(col[metric]):
            record(f"s6_{tag}_{codes[name]}_{key}", col[metric], fmt)

def fmt_cell(v, metric):
    if pd.isna(v):
        return "—"
    if metric in ("Alpha (annualised)", "Annual return", "CVaR 5% (monthly)", "Max drawdown"):
        return f"{v:.2%}"
    if metric == "Annual variance":
        return f"{v:.4f}"
    return f"{v:.2f}"

summary6_display = summary6.apply(lambda row: row.map(lambda v: fmt_cell(v, row.name)), axis = 1)
summary6_display

period                          In-sample                                    \
strategy           B Benchmark (1/N held) S1 Paper (Exhibit 9) S2 Rate rule   
Metric                                                                        
Alpha (annualised)                      —                0.51%        2.55%   
Alpha t-stat (NW)                       —                 0.16         1.31   
Beta                                    —                 1.03         0.75   
Annual return                       7.83%                8.03%        8.60%   
Annual variance                    0.0096               0.0195       0.0097   
Annual Sharpe                        0.74                 0.57         0.81   
CVaR 5% (monthly)                   5.79%                9.83%        6.24%   
Max drawdown                      -22.18%              -49.04%      -32.71%   
Turnover p.a.                        0.00                 0.44         0.38   

period                                                    Out-of-sample  \
strategy           S3 Rate + deficit S4 Ridge MV B Benchmark (1/N held)   
Metric                                                                    
Alpha (annualised)             2.20%       4.14%                      —   
Alpha t-stat (NW)               1.11        3.04                      —   
Beta                            0.78        0.50                      —   
Annual return                  8.44%       8.66%                 10.35%   
Annual variance               0.0103      0.0044                 0.0135   
Annual Sharpe                   0.78        1.17                   0.65   
CVaR 5% (monthly)              6.43%       3.72%                  6.40%   
Max drawdown                 -33.74%     -17.91%                -19.96%   
Turnover p.a.                   0.38        0.48                   0.00   

period                                                                  \
strategy           S1 Paper (Exhibit 9) S2 Rate rule S3 Rate + deficit   
Metric                                                                   
Alpha (annualised)               -2.28%       -0.90%            -0.31%   
Alpha t-stat (NW)                 -0.86        -0.39             -0.13   
Beta                               1.02         0.81              0.85   
Annual return                     7.82%        7.96%             8.89%   
Annual variance                  0.0174       0.0113            0.0124   
Annual Sharpe                      0.40         0.48              0.54   
CVaR 5% (monthly)                 7.56%        5.82%             6.12%   
Max drawdown                    -24.79%      -20.21%           -20.78%   
Turnover p.a.                      0.41         0.31              0.23   

period                          
strategy           S4 Ridge MV  
Metric                          
Alpha (annualised)      -0.78%  
Alpha t-stat (NW)        -0.43  
Beta                      0.64  
Annual return            6.93%  
Annual variance         0.0074  
Annual Sharpe             0.47  
CVaR 5% (monthly)        4.67%  
Max drawdown           -19.55%  
Turnover p.a.             0.51

In [43]:
curves =\
(
    pd
    .DataFrame(returns)[order]
    .loc[OOS]
    .add(1).cumprod()
    .rename_axis("Date")
    .reset_index()
    .melt(id_vars = "Date", var_name = "strategy", value_name = "growth")
)
fig06 =\
(
    ggplot(curves, aes(x = "Date", y = "growth", color = "strategy"))
    + geom_line(size = 1.2)
    + scale_color_manual(values = PALETTE)
    + labs(title = "Out-of-sample: growth of $1 by strategy",
           subtitle = f"{OOS[0]:%b %Y} – {OOS[-1]:%b %Y} (last 30% of the sample), after 10 bps trading costs; B restarts at equal weights",
           caption = "Source: Yahoo Finance ETF prices; Group 6 calculations",
           x = "", y = "Growth of $1", color = "")
    + theme(text = element_text(size = 14), legend_position = "right")
    + ggsize(1100, 500)
)
save_fig(fig06, "fig06_oos_growth", legend = "top")
fig06

### 6.4 Monte Carlo: the next 10 years if rates stay high or fall

How often rates switch between higher and lower is counted from history (the CVaR paper's method, Pun, Wang & Yan p.17). For each simulated month we draw a historical month from the same regime (1960–2026, §5 assets) and apply each strategy's weights.
- **Stays high:** start higher-rate.
- **Falls:** start lower-rate.
- **Always high:** a stress case with no switching.

Long-history stand-ins: equity, 10-year bonds, gold, momentum (RF + UMD). S1 is left out because private equity, high yield and real estate have no long history. Monthly rebalancing, no costs, 2,000 paths of 120 months.

In [44]:
lab = regimes["m2"].dropna()
A = pd.crosstab(lab.shift(1).dropna(), lab.iloc[1:], normalize = "index")    # A[j, k] = P(next = k | now = j)
record("mc_p_stay_high", A.loc[1.0, 1.0], "{:.1%}"); record("mc_p_stay_low", A.loc[0.0, 0.0], "{:.1%}")

pool = s5_assets.assign(reg = regimes["m2"].shift(1).reindex(s5_assets.index)).dropna()
pools = {g: pool[pool["reg"] == g][["Equity", "Bonds", "Gold", "Momentum", "Cash"]].values for g in (0.0, 1.0)}
cols = ["Equity", "Bonds", "Gold", "Momentum"]
map4 = dict(zip(S4_ASSETS, cols))
W = {  # (low-rate weights, high-rate weights) on Equity, Bonds, Gold, Momentum
    "S2 Rate rule":      ([.7, .3, 0, 0], [.5, .5, 0, 0]),
    "S3 Rate + deficit": ([.7, .3, 0, 0], [.6, .4, 0, 0]),       # assumes the deficit stays big, as today
    "S4 Ridge MV":       ([s4_lo[a] for a in S4_ASSETS], [s4_hi[a] for a in S4_ASSETS]),
}

def simulate(start, switching = True, n_paths = 2000, months = 120, seed = 600):
    g = np.random.default_rng(seed)
    out = {k: [] for k in ["B Benchmark (1/N held)", *W]}
    for _ in range(n_paths):
        s, path = start, []
        for _m in range(months):
            path.append((s, pools[s][g.integers(len(pools[s]))]))
            if switching:
                s = 1.0 if g.random() < A.loc[s, 1.0] else 0.0
        R = np.array([p[1] for p in path]); states = np.array([p[0] for p in path]); cash = R[:, 4]
        wealth = np.cumprod(1 + R[:, :4], axis = 0).mean(axis = 1)
        b = np.diff(np.concatenate([[1.0], wealth])) / np.concatenate([[1.0], wealth[:-1]])
        rets = {"B Benchmark (1/N held)": b}
        for k, (lo, hi_) in W.items():
            wt = np.where(states[:, None] == 1.0, np.array(hi_), np.array(lo))
            rets[k] = (wt * R[:, :4]).sum(axis = 1)
        for k, r in rets.items():
            x = r - cash
            out[k].append({"ann_return": np.prod(1 + r) ** (12 / months) - 1,
                           "sharpe": x.mean() / x.std() * np.sqrt(12),
                           "cvar5": -np.sort(r)[: max(1, int(0.05 * months))].mean()})
    return {k: pd.DataFrame(v) for k, v in out.items()}

mc_rows, mc_dist = [], []
for scen, start, sw in [("Rates stay high", 1.0, True), ("Rates fall", 0.0, True), ("Always high (stress)", 1.0, False)]:
    sims = simulate(start, sw)
    for k, df in sims.items():
        mc_rows.append({"scenario": scen, "strategy": k, "median annual return": df["ann_return"].median(),
                        "5th pct annual return": df["ann_return"].quantile(0.05), "median Sharpe": df["sharpe"].median(),
                        "mean CVaR 5%": df["cvar5"].mean()})
        mc_dist.append(df.assign(scenario = scen, strategy = k))
mc_table = pd.DataFrame(mc_rows).set_index(["scenario", "strategy"])
for (scen, k), row in mc_table.iterrows():
    tag = {"Rates stay high": "high", "Rates fall": "fall", "Always high (stress)": "stress"}[scen]
    code = k.split()[0].lower()
    record(f"mc_{tag}_{code}_med", row["median annual return"], "{:.1%}")
    record(f"mc_{tag}_{code}_p5", row["5th pct annual return"], "{:.1%}")
    record(f"mc_{tag}_{code}_sharpe", row["median Sharpe"], "{:.2f}")
mc_table.round(3)

median annual return  \
scenario             strategy                                       
Rates stay high      B Benchmark (1/N held)                 0.098   
                     S2 Rate rule                           0.093   
                     S3 Rate + deficit                      0.095   
                     S4 Ridge MV                            0.086   
Rates fall           B Benchmark (1/N held)                 0.095   
                     S2 Rate rule                           0.096   
                     S3 Rate + deficit                      0.097   
                     S4 Ridge MV                            0.086   
Always high (stress) B Benchmark (1/N held)                 0.111   
                     S2 Rate rule                           0.078   
                     S3 Rate + deficit                      0.081   
                     S4 Ridge MV                            0.084   

                                             5th pct annual return  \
scenario             strategy                                        
Rates stay high      B Benchmark (1/N held)                  0.059   
                     S2 Rate rule                            0.037   
                     S3 Rate + deficit                       0.035   
                     S4 Ridge MV                             0.048   
Rates fall           B Benchmark (1/N held)                  0.057   
                     S2 Rate rule                            0.038   
                     S3 Rate + deficit                       0.037   
                     S4 Ridge MV                             0.048   
Always high (stress) B Benchmark (1/N held)                  0.069   
                     S2 Rate rule                            0.029   
                     S3 Rate + deficit                       0.027   
                     S4 Ridge MV                             0.046   

                                             median Sharpe  mean CVaR 5%  
scenario             strategy                                             
Rates stay high      B Benchmark (1/N held)          0.760         0.037  
                     S2 Rate rule                    0.513         0.056  
                     S3 Rate + deficit               0.507         0.060  
                     S4 Ridge MV                     0.594         0.037  
Rates fall           B Benchmark (1/N held)          0.794         0.036  
                     S2 Rate rule                    0.571         0.058  
                     S3 Rate + deficit               0.563         0.062  
                     S4 Ridge MV                     0.650         0.037  
Always high (stress) B Benchmark (1/N held)          0.735         0.037  
                     S2 Rate rule                    0.277         0.048  
                     S3 Rate + deficit               0.280         0.056  
                     S4 Ridge MV                     0.410         0.035

In [45]:
fig07 =\
(
    ggplot(pd.concat(mc_dist).assign(strategy = lambda d: d["strategy"].str.split().str[0]),
           aes(x = "strategy", y = "ann_return", fill = "strategy"))
    + geom_boxplot(outlier_shape = None, outlier_alpha = 0, width = 0.6)
    + facet_wrap("scenario", ncol = 3)
    + scale_fill_manual(values = [PALETTE[4], PALETTE[1], PALETTE[3], PALETTE[2]])
    + scale_y_continuous(format = ".0%")
    + labs(title = "Next 10 years: annual return by strategy and rate scenario",
           subtitle = "2,000 simulated paths; regime switching counted from history; returns drawn from 1960–2026 months in the same regime",
           caption = "Source: Ken French, FRED, World Bank; Group 6 calculations",
           x = "", y = "Annualised return over 10 years")
    + theme(text = element_text(size = 14), legend_position = "none")
    + ggsize(1100, 450)
)
save_fig(fig07, "fig07_monte_carlo")
fig07

## <a id="s7">7 · Results export</a>

[↑ Contents](#top)

Writes every `record()`-ed number to `outputs/results.json`. The report, slides and script read from it (run `qf600-sync-check` afterwards).

In [46]:
Path("outputs").mkdir(exist_ok = True)
Path("outputs/results.json").write_text(json.dumps(results, indent = 2))
print(f"{len(results)} results written to outputs/results.json")

244 results written to outputs/results.json
